### MF Portfolio Analyser (LangGraph, multi-agent)

**Flow (linear):** Agent 1 parses your CAS (local code) → Agent 2 fetches everything once, up
front: NAV history (mfapi.in), scheme list, TER and peer returns (AMFI), benchmark TRI and P/E
(niftyindices), rates and macro from the `MACRO` config you type in, and optionally each AMC's
holdings Excel and factsheet → a metrics node computes your framework's numbers locally → a mask
node divides every rupee value and unit count by `K` → Agent 3 (Claude, via your Pro sign-in)
drafts the analysis → a critic (rule checks in code + local Qwen 7B) reviews it and can send it
back to Claude → amounts are multiplied back by `K` locally → final report.

**What is sent where**
- mfapi.in, AMFI, niftyindices and Exa see only scheme codes, fund / AMC / index names. Never amounts.
- Claude sees only masked numbers (rupees and units ÷ K), percentages and ratios. Never your name,
  PAN, email, phone, address, folio numbers or nominees.
- The local Qwen model sees public factsheet text and the masked draft (it runs on your machine).

## One-time setup (do these once, outside the notebook)
1. Install Ollama from https://ollama.com, then in a terminal: `ollama pull qwen2.5:7b`
2. Run the install cell below (a bash cell: on Windows run Jupyter from Git Bash or WSL, or
   copy the pip line into a terminal).
3. Sign the bundled Claude Code into your **Pro** account and get a long-lived token. In a
   terminal (not inside Jupyter, it opens a browser):
   ```
   python -c "import claude_agent_sdk, pathlib; print(pathlib.Path(claude_agent_sdk.__file__).parent / '_bundled' / 'claude')"
   <path printed above> setup-token
   ```
   On Windows the file is `claude.exe` in the same folder.
4. Create a `.env` file next to this notebook (EXA_API_KEY is only needed if ENABLE_AMC_DOCS is on):
   ```
   EXA_API_KEY=your_exa_key
   CLAUDE_CODE_OAUTH_TOKEN=the_token_from_step_3
   CAS_PASSWORD=your_cas_pdf_password
   ```
5. Put your **detailed** CAMS/KFintech CAS PDF next to this notebook and set `CAS_PDF_PATH` below.

Hardware note: written for Python 3.12, 16 GB RAM, CPU only. Qwen 7B (4-bit) uses about 5 GB
and is slow on CPU, so it is only given short snippets.

In [ ]:
# ============================== CONFIG ==============================
import os
import sys
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Load .env file immediately so os.environ can read it
load_dotenv()

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ---------------------------------------------------------------------------
# SECRETS & PATHS (Pulled from .env, never hardcoded)
# ---------------------------------------------------------------------------
CAS_PDF_PATH = os.environ.get("CAS_PDF_PATH", "my_cas.pdf")
CAS_PASSWORD = os.environ.get("CAS_PASSWORD", "")

DATA_DIR = Path(os.environ.get("DATA_DIR", "mf_data"))        # local cache of downloaded files
REPORT_PATH = Path(os.environ.get("REPORT_PATH", "mf_report.md"))  # final markdown report

# ---------------------------------------------------------------------------
# K = privacy masking divisor.
# Every rupee value AND every unit count from your CAS is divided by K before
# anything is sent to Claude, and multiplied back by K locally at the very end.
# ---------------------------------------------------------------------------
K = int(os.environ.get("PRIVACY_K", 100))

# ---------------------------------------------------------------------------
# MODEL CONFIGURATION
# ---------------------------------------------------------------------------
OLLAMA_EXTRACT_MODEL = "qwen2.5:3b"  # Fast 3B model for local PDF factsheet extraction
OLLAMA_CRITIC_MODEL  = "qwen2.5:7b"  # 7B model for the critic review
CLAUDE_MODEL = None                  # None = Pro plan default; or e.g. "sonnet" / "opus"

HISTORY_YEARS = 5                    # years of monthly NAV / TRI history to use
MAX_PEERS = 40                       # peer funds per category used for ranks
MAX_REVISIONS = 2                    # how many times the critic may send the draft back
RISK_FREE_FALLBACK = 0.065           # used only if MACRO["tbill_91d_pct"] is left empty

# Domain allowlist for Agent 2 fetcher
ALLOWED_DOMAINS = ["amfiindia.com", "portal.amfiindia.com", "mfapi.in", "niftyindices.com","nseindia.com", "fred.stlouisfed.org"]






In [ ]:
# ---------- Rates and macro: Auto-fetching with fallback ----------
import io
import pandas as pd
import datetime as dt

def fetch_auto_cpi() -> float | None:
    """Fetches India's monthly CPI from FRED without requiring an API key and calculates YoY %."""
    try:
        url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=INDCPIALLMINMEI"
        r = _session.get(url, timeout=15)
        if r.status_code == 200:
            df = pd.read_csv(io.StringIO(r.text))
            df.columns = ["date", "cpi"]
            df["cpi"] = pd.to_numeric(df["cpi"], errors="coerce")
            df = df.dropna().sort_values("date")
            
            # Need at least 13 months to compute 1-year YoY change
            if len(df) >= 13:
                latest_cpi = df.iloc[-1]["cpi"]
                prev_year_cpi = df.iloc[-13]["cpi"]
                cpi_yoy = ((latest_cpi / prev_year_cpi) - 1.0) * 100.0
                return round(float(cpi_yoy), 2)
    except Exception as e:
        log.warning("Auto CPI fetch failed, using fallback: %s", e)
    return None

def fetch_auto_fii_dii() -> dict | None:
    """Fetches the latest institutional trading activity directly from NSE India."""
    try:
        # Establish session cookies on NSE
        _session.get("https://www.nseindia.com", timeout=10)
        url = "https://www.nseindia.com/api/fiidiiTradeReact"
        r = _session.get(url, timeout=10)
        
        if r.status_code == 200:
            data = r.json()
            out = {"fii_net_month_cr": 0.0, "dii_net_month_cr": 0.0}
            for row in data:
                cat = str(row.get("category", "")).upper()
                net_val = float(str(row.get("netValue", 0)).replace(",", ""))
                if "FII" in cat or "FPI" in cat:
                    out["fii_net_month_cr"] = net_val
                elif "DII" in cat:
                    out["dii_net_month_cr"] = net_val
            return out
    except Exception as e:
        log.warning("Auto FII/DII fetch failed, using fallback: %s", e)
    return None

# ---------- Rates and macro: Self-contained with fallback ----------
import datetime as dt

# If MACRO is not yet defined in your CONFIG cell, this safe default is used automatically:
if "MACRO" not in globals():
    MACRO = {
        "as_of": "2026-10-11",
        "repo_pct": 5.50,
        "tbill_91d_pct": 5.57,
        "gsec_10y_pct": 7.29,
        "cpi_yoy_pct": None,
        "fii_net_month_cr": None,
        "dii_net_month_cr": None,
    }

def static_rates_and_macro() -> tuple[dict, dict]:
    # Safely pull MACRO from globals or fall back
    macro_data = globals().get("MACRO", MACRO)
    src = "MACRO config"
    as_of = macro_data.get("as_of", str(dt.date.today()))
    
    try:
        age = (dt.date.today() - dt.date.fromisoformat(as_of)).days
    except Exception:
        age = None
        
    note = f"as of {as_of}" + (" (STALE: over 45 days old)" if age is None or age > 45 else "")
    f = lambda v: missing(src, "not filled in MACRO") if v is None else fact(v, src, as_of, reason=note)
    
    rates = {
        "repo": f(macro_data.get("repo_pct")),
        "tbill_91d": f(macro_data.get("tbill_91d_pct")),
        "gsec_10y": f(macro_data.get("gsec_10y_pct"))
    }
    
    flows = {k: macro_data.get(k) for k in ("fii_net_month_cr", "dii_net_month_cr")}
    macro_out = {
        "cpi": f(macro_data.get("cpi_yoy_pct")),
        "fii_dii": f(flows if any(v is not None for v in flows.values()) else None)
    }
    return rates, macro_out

    
# Look-through configuration
ENABLE_AMC_DOCS = True
AUTO_FIND_AMC_SITES = True

# Known AMC Domains (Add missing ones here to prevent search lookups)
AMC_DOMAINS = {
    "ICICI Prudential Mutual Fund": "icicipruamc.com",
    "HDFC Mutual Fund": "hdfcfund.com",
    "SBI Mutual Fund": "sbimf.com",
    "Axis Mutual Fund": "axismf.com",
    "Kotak Mahindra Mutual Fund": "kotakmf.com",
    "Parag Parikh Mutual Fund": "amc.ppfas.com",
    "Nippon India Mutual Fund": "mf.nipponindiaim.com",
    "Mirae Asset Mutual Fund": "miraeassetmf.co.in",
    "UTI Mutual Fund": "utimf.com",
    "DSP Mutual Fund": "dspim.com",
    "Groww Mutual Fund": "growwmf.in",
    "Quant Mutual Fund": "quantmutual.com",
    "Motilal Oswal Mutual Fund": "motilaloswalmf.com",
    "Tata Mutual Fund": "tatamutualfund.com",
}

# Benchmark overrides (optional, leave empty for smart auto-resolution)
BENCHMARK_MAP = {}
DEFAULT_BENCHMARK = "NIFTY 500"
MARKET_INDEX = "NIFTY 50"

# Framework thresholds
FRAMEWORK = {
    "glide_path_equity": [(3, 0.10), (5, 0.40), (7, 0.60), (99, 0.80)],
    "ideal_fund_count": (5, 8),
    "overlap_redundant": 0.50,
    "amc_concentration_max": 0.35,
    "rebalance_band_abs": 0.05,
    "rebalance_band_rel": 0.25,
    "macro_tilt_max": 0.10,
    "info_ratio_good": 0.5, "info_ratio_excellent": 0.75,
    "r2_closet_indexer": 0.90, "active_share_min": 0.60,
    "exit_bottom_quartile_quarters": 4,
    "watch_below_median_quarters": 2,
    "scorecard_weights": {"consistency": 0.25, "risk_adjusted": 0.25, "downside": 0.15,
                          "cost": 0.10, "manager": 0.15, "fit": 0.10},
    "tax_rules_as_of": "2026-10-11",
    "equity_ltcg_days": 365, "equity_stcg_rate": 0.20, "equity_ltcg_rate": 0.125,
    "equity_ltcg_exemption": 125000, "debt_slab_rate": 0.30,
    "default_exit_load": {"equity": (365, 0.01), "hybrid": (365, 0.01), "debt": (0, 0.0)},
    "mc_assumptions": {"equity": (0.11, 0.18), "debt": (0.07, 0.03), "gold": (0.08, 0.15), "intl": (0.10, 0.18)},
    "mc_runs": 10000,
}

# Client profile
PROFILE = {
    "age": 35, "dependents": 1, "income_stability": "stable",
    "emergency_fund_months": 6, "debt_to_income": 0.20,
    "risk_tolerance_score": 6,
    "stopped_sip_in_2020_or_2022": False,
    "horizon_years": 10,
    "monthly_sip": None,
}

In [ ]:
# ============================== COMMON HELPERS ==============================
import os, re, io, json, math, time, copy, uuid, gzip, asyncio, logging, datetime as dt
from urllib.parse import urlparse
from typing import Any, TypedDict

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
log = logging.getLogger("mf")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36",
    "Accept": "*/*", "Accept-Language": "en-US,en;q=0.9",
}

_found_amc_path = DATA_DIR / "amc_domains_found.json"
AMC_FOUND = json.loads(_found_amc_path.read_text()) if _found_amc_path.exists() else {}

def domain_allowed(url: str) -> bool:
    """Agent 2's hard allowlist: listed sites + your AMC sites (configured or found). Nothing else is fetched."""
    host = (urlparse(url).hostname or "").lower()
    allowed = ALLOWED_DOMAINS + list(AMC_DOMAINS.values()) + list(AMC_FOUND.values())
    return any(host == d or host.endswith("." + d) for d in allowed)

_session = requests.Session()
_session.headers.update(BROWSER_HEADERS)

def http(method: str, url: str, retries: int = 3, **kw) -> requests.Response:
    if not domain_allowed(url):
        raise PermissionError(f"Blocked by allowlist: {url}")
    kw.setdefault("timeout", 60)
    last = None
    for attempt in range(retries):
        try:
            r = _session.request(method, url, **kw)
            r.raise_for_status()
            return r
        except Exception as e:  # network errors and 5xx: back off and retry
            last = e
            time.sleep(2 ** attempt)
    raise last

def cache_path(*parts: str) -> Path:
    p = DATA_DIR.joinpath(*parts)
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

def fact(value, source_url: str, as_of=None, status: str = "ok", reason: str = "") -> dict:
    """Every fetched number carries where it came from and whether it is trustworthy."""
    return {"value": value, "source_url": source_url, "as_of": str(as_of) if as_of else None,
            "status": status, "reason": reason}

def missing(source_url: str, reason: str) -> dict:
    return fact(None, source_url, status="missing", reason=reason)

def norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

## Agent 1: CAS parser (local code, no LLM)
Uses `casparser`, a regex-based parser for CAMS / KFintech detailed CAS PDFs. Produces:
- **o1** (to Agent 2): fund identity only. No units, no amounts, no folio, no personal data.
- **o2** (towards Agent 3): holdings and transactions. Kept with real values locally; the mask
  node divides rupees and units by K before Claude sees it. Personal data is dropped here.

In [ ]:
# ============================== AGENT 1: CAS PARSER ==============================
import re
import uuid
import casparser

EQUITY_WORDS = ("equity", "elss", "large cap", "mid cap", "small cap", "flexi", "multi cap",
                "focused", "value", "contra", "dividend yield", "sectoral", "thematic", "index", "etf")

def _plan(name: str) -> str:
    return "direct" if re.search(r"\bdirect\b", name, re.I) else "regular"

def _option(name: str) -> str:
    return "idcw" if re.search(r"idcw|dividend|payout|reinvest", name, re.I) else "growth"

def _clean_amc(amc: str | None, scheme_name: str) -> str:
    """If CAS reports UNKNOWN or None for AMC, infer it from the scheme name."""
    if amc and str(amc).strip().upper() not in ("UNKNOWN", "NONE", "", "NAN"):
        return str(amc).strip()

    # Pre-defined list of common AMCs to match against scheme name
    amc_candidates = list(globals().get("AMC_DOMAINS", {}).keys()) + [
        "Parag Parikh Mutual Fund", "HDFC Mutual Fund", "ICICI Prudential Mutual Fund",
        "SBI Mutual Fund", "Axis Mutual Fund", "Kotak Mahindra Mutual Fund",
        "Nippon India Mutual Fund", "Mirae Asset Mutual Fund", "UTI Mutual Fund",
        "DSP Mutual Fund", "Quant Mutual Fund", "Motilal Oswal Mutual Fund",
        "Tata Mutual Fund", "Bandhan Mutual Fund", "Canara Robeco Mutual Fund",
        "Franklin Templeton Mutual Fund", "Invesco Mutual Fund", "Sundaram Mutual Fund",
        "Edelweiss Mutual Fund", "HSBC Mutual Fund", "360 ONE Mutual Fund",
        "Groww Mutual Fund", "Zerodha Mutual Fund", "Navi Mutual Fund", "WhiteOak Capital Mutual Fund"
    ]

    norm_scheme = scheme_name.lower()
    for candidate in amc_candidates:
        # Match using brand name (e.g. 'Parag Parikh', 'HDFC', 'ICICI Prudential')
        brand = re.sub(r"\bmutual fund\b", "", candidate, flags=re.I).strip().lower()
        if brand and brand in norm_scheme:
            return candidate

    # Generic fallback: take the first two words from the scheme name
    words = [w for w in re.split(r"[\s\-_]+", scheme_name) if w]
    prefix = " ".join(words[:2]) if len(words) >= 2 else (words[0] if words else "Unknown")
    return f"{prefix} Mutual Fund"

def parse_cas(pdf_path: str, password: str) -> tuple[list[dict], list[dict], dict]:
    data = casparser.read_cas_pdf(pdf_path, password)
    o1, o2, folio_ids = [], [], {}
    
    for folio in data.folios:
        # Folio numbers are personal: replace with a stable anonymous ID
        fid = folio_ids.setdefault(folio.folio, "F" + uuid.uuid4().hex[:6])
        
        for s in folio.schemes:
            if not s.isin or float(s.close) <= 0:
                continue  # skip closed/zero-unit holdings
            
            # Resolve AMC accurately even if casparser parsed it as UNKNOWN
            amc_name = _clean_amc(getattr(folio, "amc", None), s.scheme)
            
            txns = []
            for t in s.transactions:
                txns.append({
                    "date": str(t.date),
                    "type": str(t.type.value if hasattr(t.type, "value") else t.type),
                    "amount": float(t.amount) if t.amount is not None else None,
                    "units": float(t.units) if t.units is not None else None,
                    "nav": float(t.nav) if t.nav is not None else None,
                })
                
            o2.append({
                "folio_id": fid,
                "isin": s.isin,
                "amfi_code": s.amfi,
                "scheme_name": s.scheme,
                "amc": amc_name,
                "units": float(s.close),
                "open_units": float(s.open),
                "nav": float(s.valuation.nav),
                "nav_date": str(s.valuation.date),
                "value": float(s.valuation.value),
                "cost": float(s.valuation.cost) if s.valuation.cost is not None else None,
                "transactions": txns,
            })
            
            if s.isin not in {f["isin"] for f in o1}:
                o1.append({
                    "isin": s.isin,
                    "amfi_code": s.amfi,
                    "scheme_name": s.scheme,
                    "amc": amc_name,
                    "plan": _plan(s.scheme),
                    "option": _option(s.scheme)
                })

    meta = {
        "statement_period": {
            "from": data.statement_period.from_,
            "to": data.statement_period.to
        },
        "parse_warnings": list(getattr(data, "parse_warnings", []) or [])
    }
    
    reconcile(o2)
    return o1, o2, meta

def reconcile(o2: list[dict]) -> None:
    """Units from transactions must add up to closing units, and units x NAV must match value."""
    for h in o2:
        txn_units = h["open_units"] + sum(t["units"] or 0 for t in h["transactions"])
        if h["transactions"] and abs(txn_units - h["units"]) > 0.01 * max(1, h["units"]):
            log.warning("Units mismatch for %s: opening + transactions %.3f vs closing %.3f", 
                        h["scheme_name"], txn_units, h["units"])
        if abs(h["units"] * h["nav"] - h["value"]) > 0.01 * max(1, h["value"]):
            raise ValueError(f"Valuation mismatch for {h['scheme_name']}")

## Agent 2: data fetchers (fixed Python, runs once, up front)
Fixed Python tools, one per source, all behind the domain allowlist. Funds are matched by exact
ISIN (AMFI scheme list), names by token overlap. Exa and the local Qwen model are used only by
the optional AMC documents module (holdings Excel, factsheet PDF), Qwen with a strict JSON schema. Every value carries `source_url`, `as_of` and `status`; failures become
`status="missing"` with a reason, never a guessed number. Only the fields the analysis needs
are kept from each source.

In [ ]:
# ---------- NAVs: AMFI latest file (scheme list + categories) and mfapi.in (history per fund) ----------
NAV_LATEST_URL = "https://portal.amfiindia.com/spages/NAVAll.txt"
MFAPI_URL = "https://api.mfapi.in/mf/{code}"

def parse_amfi_nav_text(text: str) -> pd.DataFrame:
    """Parse AMFI's semicolon NAV text (latest or history). Section lines give the SEBI category."""
    rows, category, cols = [], None, None
    for raw in text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if ";" not in line:
            m = re.search(r"Ended Schemes?\s*\((.*)\)", line, re.I)
            if m:
                category = m.group(1).strip()
            continue
        parts = [p.strip() for p in line.split(";")]
        if cols is None or parts[0].lower().startswith("scheme code"):
            if parts[0].lower().startswith("scheme code"):
                cols = [p.lower() for p in parts]
            continue
        if not parts[0].isdigit():
            continue
        get = lambda key: next((parts[i] for i, c in enumerate(cols) if key in c and i < len(parts)), None)
        nav = get("net asset value")
        try:
            nav = float(nav)
        except (TypeError, ValueError):
            continue
        rows.append({
            "code": parts[0], "name": get("scheme name") or get("nav name"),
            "isin_growth": get("isin div payout"), "isin_reinv": get("reinvest"),
            "nav": nav, "date": pd.to_datetime(get("date"), format="%d-%b-%Y", errors="coerce"),
            "category": category,
        })
    return pd.DataFrame(rows)

def fetch_latest_navs() -> pd.DataFrame:
    p = cache_path("nav", f"latest_{dt.date.today()}.txt.gz")
    if not p.exists():
        p.write_bytes(gzip.compress(http("GET", NAV_LATEST_URL).content))
    return parse_amfi_nav_text(gzip.decompress(p.read_bytes()).decode("utf-8", "ignore"))

def fetch_scheme_meta() -> pd.DataFrame:
    """One call: every open scheme's code, name, ISINs and SEBI category (AMFI's latest NAV file)."""
    df = fetch_latest_navs().drop_duplicates("code").set_index("code")
    return df[["name", "isin_growth", "isin_reinv", "category"]]

def fetch_nav_history(code: str) -> pd.Series | None:
    """Full daily NAV history for one scheme from mfapi.in (one call, cached for the day)."""
    p = cache_path("nav", f"{code}_{dt.date.today()}.json.gz")
    if not p.exists():
        for old in p.parent.glob(f"{code}_*.json.gz"):
            old.unlink()   # keep only today's copy
        p.write_bytes(gzip.compress(http("GET", MFAPI_URL.format(code=code)).content))
    rows = json.loads(gzip.decompress(p.read_bytes())).get("data") or []
    navs = {}
    for r in rows:
        try:
            navs[pd.to_datetime(r["date"], format="%d-%m-%Y")] = float(r["nav"])
        except (KeyError, TypeError, ValueError):
            continue
    return pd.Series(navs).sort_index() if navs else None

def build_month_end_panel(codes: list[str], years: int = HISTORY_YEARS) -> pd.DataFrame:
    """Month-end NAV for your funds and their peers (columns = AMFI scheme codes)."""
    today = pd.Timestamp(dt.date.today())
    start = today - pd.DateOffset(years=years, months=1)
    cols = {}
    for c in codes:
        try:
            s = fetch_nav_history(c)
        except Exception as e:
            log.warning("NAV history for %s failed: %s", c, e)
            continue
        if s is not None and len(s):
            cols[c] = s[s.index >= start].resample("ME").last()
    panel = pd.DataFrame(cols).sort_index()
    return panel[panel.index <= today]   # drop the current, unfinished month

def classify_category(category: str | None) -> tuple[str, str]:
    """'Equity Scheme - Large Cap Fund' -> ('equity', 'Large Cap Fund')."""
    if not category:
        return "unknown", "unknown"
    head, _, sub = category.partition(" - ")
    h = head.lower()
    asset = ("equity" if "equity" in h else "debt" if "debt" in h else "hybrid" if "hybrid" in h
             else "solution" if "solution" in h else "other")
    sub = sub.strip() or head.strip()
    if asset == "other" and re.search(r"gold", sub, re.I):
        asset = "gold"
    elif asset == "other" and re.search(r"overseas|fof.*(international|global)|international", sub, re.I):
        asset = "intl"
    elif asset == "other" and re.search(r"index|etf", sub, re.I):
        asset = "equity"
    return asset, sub

In [ ]:
# ---------- AMFI Fund Performance (trailing returns + full peer set per sub-category) ----------
from amfipy import AMFIClient   # amfiindia.com only

_amfi = AMFIClient()

def _pick(d: dict, *keys):
    """Case/format-insensitive key lookup (AMFI's JSON keys vary)."""
    nd = {norm(k).replace(" ", ""): v for k, v in d.items()}
    for k in keys:
        v = nd.get(norm(k).replace(" ", ""))
        if v not in (None, "", "-"):
            return v
    return None

AMFI_PERF_CATEGORY = {"equity": 1, "debt": 2, "hybrid": 3, "solution": 4, "other": 5, "gold": 5, "intl": 5}

def fetch_fund_performance(asset: str, sub_category: str) -> dict:
    src = "https://www.amfiindia.com/otherdata/fund-performance"
    try:
        cat_id = AMFI_PERF_CATEGORY.get(asset, 5)
        subs = _amfi.fund_performance.sub_categories(cat_id)
        best = max(subs, key=lambda s: _overlap(sub_category, str(_pick(s, "name", "subCategoryName", "subcategory") or "")))
        sub_id = _pick(best, "id", "subCategoryId", "subcategoryid")
        for back in range(6):  # report date must be a business day
            d = (dt.date.today() - dt.timedelta(days=back)).strftime("%d-%b-%Y")
            recs = _amfi.fund_performance.fetch(category=cat_id, sub_category=int(sub_id), report_date=d)
            if recs:
                keep = []
                for r in recs:  # keep only the fields the analysis needs
                    keep.append({
                        "scheme": _pick(r, "schemeName", "Scheme_Name"),
                        "benchmark": _pick(r, "benchmark", "benchmarkName"),
                        "ret_1y_reg": _num(_pick(r, "return1YearRegular", "Returns_1yr", "oneYearRegular")),
                        "ret_1y_dir": _num(_pick(r, "return1YearDirect", "oneYearDirect")),
                        "ret_3y_reg": _num(_pick(r, "return3YearRegular", "Returns_3yr", "threeYearRegular")),
                        "ret_3y_dir": _num(_pick(r, "return3YearDirect", "threeYearDirect")),
                        "ret_5y_reg": _num(_pick(r, "return5YearRegular", "Returns_5yr", "fiveYearRegular")),
                        "ret_5y_dir": _num(_pick(r, "return5YearDirect", "fiveYearDirect")),
                        "bench_1y": _num(_pick(r, "benchmark1Year", "Benchmark_Returns_1yr", "return1YearBenchmark")),
                        "bench_3y": _num(_pick(r, "benchmark3Year", "Benchmark_Returns_3yr", "return3YearBenchmark")),
                        "bench_5y": _num(_pick(r, "benchmark5Year", "Benchmark_Returns_5yr", "return5YearBenchmark")),
                    })
                return fact(keep, src, d)
        return missing(src, "no records for recent business days")
    except Exception as e:
        return missing(src, f"fund performance fetch failed: {e}")

def _num(x):
    try:
        return float(str(x).replace(",", "").replace("%", ""))
    except (TypeError, ValueError):
        return None

STOP_WORDS = {"fund", "plan", "direct", "regular", "growth", "option", "idcw", "dividend", "the", "scheme",
              "payout", "reinvestment", "formerly", "known", "as", "mutual"}

def _tokens(s: str) -> set:
    return {t for t in norm(s).split() if t not in STOP_WORDS}

def _overlap(a: str, b: str) -> float:
    ta, tb = _tokens(a), _tokens(b)
    return len(ta & tb) / max(1, len(ta)) if ta else 0.0

In [ ]:
# ---------- AMFI TER (Patched) ----------
def fetch_ter(scheme_name: str) -> dict:
    src = "https://www.amfiindia.com/ter-of-mf-schemes"
    try:
        today = dt.date.today()
        p = cache_path("ter", f"{today:%Y-%m}.json")
        recs = []
        if p.exists():
            try:
                recs = json.loads(p.read_text())
            except Exception:
                pass
        
        # If cache doesn't exist or is empty, attempt fetch
        if not recs:
            try:
                fy = f"{today.year}-{today.year + 1}" if today.month >= 4 else f"{today.year - 1}-{today.year}"
                recs = _amfi.ter.fetch(year=fy) or []
                if recs:
                    p.write_text(json.dumps(recs, default=str))
            except Exception:
                recs = []

        if not recs:
            return missing(src, "AMFI TER API temporarily unavailable")

        best, score = None, 0.0
        for r in recs:
            sc = _overlap(scheme_name, str(_pick(r, "SchemeName", "Scheme_Name", "schemename") or ""))
            if sc > score:
                best, score = r, sc
        if not best or score < 0.6:
            return missing(src, "scheme not found in TER table")

        direct = _num(_pick(best, "D_TotalTER", "DirectTotalTER", "TotalTERDirect", "Direct_Total_TER"))
        regular = _num(_pick(best, "R_TotalTER", "RegularTotalTER", "TotalTERRegular", "Regular_Total_TER"))
        return fact({"direct_pct": direct, "regular_pct": regular}, src, _pick(best, "Month", "TERDate"))
    except Exception as e:
        return missing(src, f"TER fetch failed: {e}")

In [ ]:

# ---------- niftyindices: TRI, P/E-P/B history (Patched) ----------
NIFTY_HEADERS = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Content-Type": "application/json; charset=UTF-8",
    "Origin": "https://www.niftyindices.com",
    "Referer": "https://www.niftyindices.com/reports/historical-data",
    "X-Requested-With": "XMLHttpRequest",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0 Safari/537.36",
}

def _nifty_post(endpoint: str, index: str, start: dt.date, end: dt.date) -> pd.DataFrame:
    # Bypass the warm-up GET request completely to prevent the ?ReturnUrl= redirect loop
    cinfo = ("{'name':'%s','startDate':'%s','endDate':'%s','indexName':'%s'}"
             % (index, start.strftime("%d-%b-%Y"), end.strftime("%d-%b-%Y"), index))
    
    # Send cold POST with allow_redirects=False
    url = f"https://www.niftyindices.com/Backpage.aspx/{endpoint}"
    r = _session.post(url, json={"cinfo": cinfo}, headers=NIFTY_HEADERS, timeout=30, allow_redirects=False)
    
    if r.status_code != 200:
        url_alt = f"https://www.niftyindices.com/BackPage/{endpoint}"
        r = _session.post(url_alt, json={"cinfo": cinfo}, headers=NIFTY_HEADERS, timeout=30, allow_redirects=False)

    r.raise_for_status()
    payload = r.json()
    if isinstance(payload, dict) and "d" in payload:
        payload = json.loads(payload["d"])
    return pd.DataFrame.from_records(payload)

def _nifty_series(endpoint: str, index: str, years: int, value_pick) -> pd.DataFrame:
    end = dt.date.today()
    start = end - dt.timedelta(days=365 * years + 40)
    p = cache_path("nifty", f"{endpoint}_{norm(index).replace(' ', '_')}_{end}.csv")
    if p.exists():
        return pd.read_csv(p, index_col=0, parse_dates=True)
    frames, s = [], start
    while s < end:  # one year per request
        e = min(end, s + dt.timedelta(days=364))
        frames.append(_nifty_post(endpoint, index, s, e))
        s = e + dt.timedelta(days=1)
    raw = pd.concat(frames, ignore_index=True)
    date_col = next(c for c in raw.columns if "date" in c.lower())
    out = value_pick(raw)
    out.index = pd.to_datetime(raw[date_col], dayfirst=True, errors="coerce")
    out = out[~out.index.isna()].sort_index()
    out = out[~out.index.duplicated(keep="last")]
    out.to_csv(p)
    return out

def fetch_tri(index: str, years: int = HISTORY_YEARS) -> dict:
    src = "https://www.niftyindices.com/reports/historical-data"
    def pick(raw):
        cols = [c for c in raw.columns if re.search(r"total.?return|tri", c, re.I) and not re.search(r"net", c, re.I)]
        col = cols[0] if cols else raw.columns[-1]
        return pd.to_numeric(raw[col].astype(str).str.replace(",", ""), errors="coerce").to_frame("tri")
    try:
        s = _nifty_series("getTotalReturnIndexString", index, years, pick)["tri"].dropna()
        if len(s) < 200 or (s <= 0).any():
            return missing(src, f"TRI for {index} incomplete")
        return fact(s, src, s.index[-1].date())
    except Exception as e:
        return missing(src, f"TRI fetch failed for {index}: {e}")

def fetch_pe_pb(index: str = MARKET_INDEX, years: int = 10) -> dict:
    src = "https://www.niftyindices.com/reports/historical-data"
    def pick(raw):
        out = {}
        for c in raw.columns:
            cl = c.lower().replace(" ", "")
            if cl in ("pe", "p/e"): out["pe"] = raw[c]
            if cl in ("pb", "p/b"): out["pb"] = raw[c]
            if "div" in cl: out["div_yield"] = raw[c]
        return pd.DataFrame({k: pd.to_numeric(v, errors="coerce") for k, v in out.items()})
    try:
        df = _nifty_series("getpepbHistoricaldataDBtoString", index, years, pick).dropna(how="all")
        return fact(df, src, df.index[-1].date())
    except Exception as e:
        return missing(src, f"P/E-P/B fetch failed: {e}")

In [ ]:
# ---------- Rates and macro: from the MACRO config (no scrapers) ----------
def static_rates_and_macro() -> tuple[dict, dict]:
    src = "MACRO in the config cell (typed in by you)"
    as_of = MACRO.get("as_of")
    age = (dt.date.today() - dt.date.fromisoformat(as_of)).days if as_of else None
    note = f"typed in, as of {as_of}" + (" (STALE: over 45 days old, update MACRO)" if age is None or age > 45 else "")
    f = lambda v: missing(src, "not filled in MACRO") if v is None else fact(v, src, as_of, reason=note)
    rates = {"repo": f(MACRO.get("repo_pct")), "tbill_91d": f(MACRO.get("tbill_91d_pct")),
             "gsec_10y": f(MACRO.get("gsec_10y_pct"))}
    flows = {k: MACRO.get(k) for k in ("fii_net_month_cr", "dii_net_month_cr")}
    macro = {"cpi": f(MACRO.get("cpi_yoy_pct")),
             "fii_dii": f(flows if any(v is not None for v in flows.values()) else None)}
    return rates, macro

In [ ]:
# ---------- Exa search (finds file/page links on allowed domains only) ----------
_exa = None

def exa_find(query: str, domains: list[str], n: int = 8, days_back: int | None = 75) -> list[str]:
    global _exa
    if _exa is None:
        from exa_py import Exa
        _exa = Exa(api_key=os.environ["EXA_API_KEY"])
    kw = {"num_results": n, "include_domains": domains}
    if days_back:
        kw["start_published_date"] = (dt.date.today() - dt.timedelta(days=days_back)).isoformat()
    try:
        res = _exa.search(query, **kw)
    except Exception:
        kw.pop("start_published_date", None)   # many files have no publish date: retry without it
        res = _exa.search(query, **kw)
    return [r.url for r in res.results if domain_allowed(r.url)]

AGGREGATOR_SITES = ("valueresearchonline", "morningstar", "moneycontrol", "groww", "etmoney", "kuvera",
                    "paytmmoney", "zerodha", "coin.", "mfapi", "advisorkhoj", "economictimes", "wikipedia",
                    "amfiindia", "sebi.gov", "bank")

def amc_domain(amc: str) -> str | None:
    """The AMC's own website: from AMC_DOMAINS (loose name match), then earlier finds, then an Exa lookup."""
    best = max(AMC_DOMAINS, key=lambda k: _overlap(k, amc), default=None)
    if best and _overlap(best, amc) >= 0.99:   # every word of the configured name appears in the CAS name
        return AMC_DOMAINS[best]
    if amc in AMC_FOUND:
        return AMC_FOUND[amc]
    if not AUTO_FIND_AMC_SITES:
        return None
    from exa_py import Exa
    words = [t for t in _tokens(amc) if t not in {"india", "asset", "management", "amc", "ltd", "limited", "co"}]
    try:
        res = Exa(api_key=os.environ["EXA_API_KEY"]).search(f"{amc} mutual fund official website", num_results=8)
    except Exception as e:
        log.warning("AMC site lookup failed for %s: %s", amc, e)
        return None
    scored = []
    for r in res.results:
        host = (urlparse(r.url).hostname or "").lower().removeprefix("www.")
        if not host or any(a in host for a in AGGREGATOR_SITES):
            continue
        name_hits = sum(1 for w in words if w in host.replace("-", ""))
        if name_hits == 0:
            continue   # the site's own name must contain the AMC's name
        mf_hint = any(x in host for x in ("mf", "mutual", "amc", "fund", "asset", "invest"))
        scored.append((name_hits + (1 if mf_hint else 0), host))
    if not scored:
        log.warning("No official site found for %s: holdings and factsheet will be marked missing.", amc)
        return None
    host = max(scored)[1]
    AMC_FOUND[amc] = host
    _found_amc_path.write_text(json.dumps(AMC_FOUND, indent=1))
    log.warning("Found %s for '%s' and added it to the allowlist. Check it; add it to AMC_DOMAINS to keep it.", host, amc)
    return host

def links_on_page(url: str, pattern: str) -> list[str]:
    soup = BeautifulSoup(http("GET", url).text, "lxml")
    out = []
    for a in soup.find_all("a", href=True):
        href = requests.compat.urljoin(url, a["href"])
        if re.search(pattern, href + " " + a.get_text(" "), re.I) and domain_allowed(href):
            out.append(href)
    return out

In [ ]:
# ---------- Local Qwen extraction (only for text code cannot parse) ----------
def qwen_extract(text: str, schema: dict, instruction: str) -> dict:
    import ollama
    resp = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "system", "content": "Extract only facts stated in the text. "
                                                "Use null when a value is not stated. Never guess or compute."},
                  {"role": "user", "content": f"{instruction}\n\nTEXT:\n{text[:6000]}"}],
        format=schema, options={"temperature": 0, "num_ctx": 8192},
    )
    return json.loads(resp.message.content)

In [ ]:
# ---------- Monthly portfolio Excel (full holdings) ----------
import zipfile

def _last_month() -> dt.date:
    first = dt.date.today().replace(day=1)
    return first - dt.timedelta(days=1)

def find_portfolio_files(amc: str) -> list[str]:
    domain = amc_domain(amc)
    if not domain:
        return []
    m = _last_month()
    month_words = f"{m:%B} {m:%Y}"
    urls = exa_find(f"{amc} monthly portfolio disclosure {month_words} excel", [domain])
    files = [u for u in urls if re.search(r"\.(xlsx?|zip)(\?|$)", u, re.I)]
    for page in [u for u in urls if u not in files][:3]:   # result is a listing page: scan its links
        try:
            files += links_on_page(page, rf"\.(xlsx?|zip)(\?|$).*|{m:%B}.*{m:%Y}.*\.(xlsx?|zip)")
        except Exception:
            pass
    # Prefer files whose URL mentions last month
    files = sorted(set(files), key=lambda u: -(f"{m:%b}".lower() in u.lower() or f"{m:%m}{m:%Y}" in u))
    return files

def _read_workbooks(content: bytes, url: str) -> dict[str, pd.DataFrame]:
    """Return {sheet_name: raw DataFrame} from an xls/xlsx or a zip of them."""
    books = {}
    if url.lower().split("?")[0].endswith(".zip") or content[:2] == b"PK" and b"[Content_Types].xml" not in content[:2000]:
        try:
            with zipfile.ZipFile(io.BytesIO(content)) as z:
                for name in z.namelist():
                    if re.search(r"\.xlsx?$", name, re.I):
                        for sh, df in pd.read_excel(io.BytesIO(z.read(name)), sheet_name=None, header=None).items():
                            books[f"{name}:{sh}"] = df
                return books
        except zipfile.BadZipFile:
            pass
    return pd.read_excel(io.BytesIO(content), sheet_name=None, header=None)

def parse_holdings_sheet(raw: pd.DataFrame) -> pd.DataFrame | None:
    """Generic parser: find the header row by column names, keep only the needed columns."""
    for i in range(min(40, len(raw))):
        cells = [norm(c) for c in raw.iloc[i].tolist()]
        if any(c == "isin" or c.startswith("isin") for c in cells) and any(
                "nav" in c or "net assets" in c for c in cells):
            col = lambda pred: next((j for j, c in enumerate(cells) if pred(c)), None)
            c_isin = col(lambda c: c.startswith("isin"))
            c_w = col(lambda c: ("nav" in c or "net assets" in c) and "%" in str(raw.iloc[i].tolist()) or c.startswith("to nav") or "to net assets" in c)
            c_w = c_w if c_w is not None else col(lambda c: "nav" in c or "net assets" in c)
            c_name = col(lambda c: "instrument" in c or "name of" in c or "company" in c or "issuer" in c)
            c_ind = col(lambda c: "industry" in c or "rating" in c or "sector" in c)
            body = raw.iloc[i + 1:]
            df = pd.DataFrame({
                "instrument": body.iloc[:, c_name] if c_name is not None else None,
                "isin": body.iloc[:, c_isin].astype(str).str.strip(),
                "industry_or_rating": body.iloc[:, c_ind] if c_ind is not None else None,
                "weight": pd.to_numeric(body.iloc[:, c_w], errors="coerce"),
            })
            df = df[df["isin"].str.match(r"^IN[A-Z0-9]{10}$") & df["weight"].notna()]
            if df.empty:
                return None
            if df["weight"].max() <= 1.0:   # some AMCs give fractions instead of percent
                df["weight"] *= 100
            return df.reset_index(drop=True)
    return None

def fetch_holdings(scheme_name: str, amc: str) -> dict:
    files = find_portfolio_files(amc)
    if not files:
        return missing(f"https://{amc_domain(amc) or '?'}", "monthly portfolio file not found (check AMC_DOMAINS)")
    for url in files[:4]:
        try:
            p = cache_path("portfolio", re.sub(r"[^A-Za-z0-9.]+", "_", url)[-120:])
            if not p.exists():
                p.write_bytes(http("GET", url).content)
            books = _read_workbooks(p.read_bytes(), url)
        except Exception as e:
            log.warning("portfolio file %s failed: %s", url, e)
            continue
        best, best_score = None, 0.0
        for sheet, raw in books.items():
            head = " ".join(str(x) for x in raw.head(8).values.ravel() if pd.notna(x)) + " " + sheet
            score = _overlap(scheme_name, head)
            if score > best_score:
                best, best_score = raw, score
        if best is not None and best_score >= 0.6:
            h = parse_holdings_sheet(best)
            if h is not None:
                total = h["weight"].sum()
                status = "ok" if 80 <= total <= 105 else "missing"
                return fact(h.to_dict("records"), url, _last_month(), status=status,
                            reason="" if status == "ok" else f"weights sum to {total:.1f}%")
    return missing(files[0], "scheme sheet not found in portfolio files")

In [ ]:
# ---------- Factsheet facts (manager, exit load, debt metrics) via pdfplumber + Qwen 3B + Cache ----------
import re
import json
import datetime as dt
import pdfplumber

FACTSHEET_SCHEMA = {
    "type": "object",
    "properties": {
        "fund_manager": {"type": ["string", "null"]},
        "manager_since": {"type": ["string", "null"]},
        "benchmark": {"type": ["string", "null"]},
        "exit_load_days": {"type": ["number", "null"]},
        "exit_load_pct": {"type": ["number", "null"]},
        "ytm_pct": {"type": ["number", "null"]},
        "modified_duration_yrs": {"type": ["number", "null"]},
        "macaulay_duration_yrs": {"type": ["number", "null"]},
        "avg_maturity_yrs": {"type": ["number", "null"]},
        "aum_cr": {"type": ["number", "null"]},
    },
    "required": ["fund_manager", "manager_since", "benchmark", "exit_load_days", "exit_load_pct",
                 "ytm_pct", "modified_duration_yrs", "macaulay_duration_yrs", "avg_maturity_yrs", "aum_cr"],
}

def qwen_extract(text: str, schema: dict, instruction: str) -> dict:
    """Uses the 3B model by default for extraction to maintain fast CPU speeds."""
    import ollama
    model_name = globals().get("OLLAMA_EXTRACT_MODEL", "qwen2.5:3b")
    resp = ollama.chat(
        model=model_name,
        messages=[
            {"role": "system", "content": "Extract only facts stated in the text. "
                                          "Use null when a value is not stated. Never guess or compute."},
            {"role": "user", "content": f"{instruction}\n\nTEXT:\n{text[:6000]}"}
        ],
        format=schema,
        options={"temperature": 0, "num_ctx": 8192},
    )
    return json.loads(resp.message.content)

def _get_target_months() -> list[dt.date]:
    first_this_month = dt.date.today().replace(day=1)
    last_m = first_this_month - dt.timedelta(days=1)
    month_before = last_m.replace(day=1) - dt.timedelta(days=1)
    return [last_m, month_before]

def fetch_factsheet_facts(scheme_name: str, amc: str) -> dict:
    domain = amc_domain(amc)
    if not domain:
        return missing("", f"no website known or found for {amc}")
    
    scheme_slug = re.sub(r"[^a-z0-9]+", "_", scheme_name.lower()).strip("_")
    candidate_months = _get_target_months()

    # 1. Check local JSON cache first (Instant return if already parsed)
    for m in candidate_months:
        facts_cache_file = cache_path("factsheet_facts", f"{scheme_slug}_{m:%Y_%m}.json")
        if facts_cache_file.exists():
            try:
                cached_payload = json.loads(facts_cache_file.read_text(encoding="utf-8"))
                return fact(cached_payload["data"], cached_payload["source_url"], m)
            except Exception:
                pass

    # 2. Search online with fallback across last 2 months
    for m in candidate_months:
        urls = exa_find(f"{amc} factsheet {m:%B} {m:%Y} pdf", [domain])
        pdfs = [u for u in urls if re.search(r"\.pdf(\?|$)", u, re.I)]
        for page in [u for u in urls if u not in pdfs][:2]:
            try:
                pdfs += links_on_page(page, r"factsheet.*\.pdf|\.pdf.*factsheet")
            except Exception:
                pass

        for url in pdfs[:3]:
            try:
                clean_url_stem = re.sub(r"[^A-Za-z0-9.]+", "_", url)[-120:]
                pdf_path = cache_path("factsheet", clean_url_stem)
                txt_cache_path = cache_path("factsheet", clean_url_stem + ".txt")

                # Download PDF if not present
                if not pdf_path.exists():
                    pdf_path.write_bytes(http("GET", url).content)

                # Extract text using pdfplumber ONCE per PDF file and cache it
                if txt_cache_path.exists():
                    pdf_text = txt_cache_path.read_text(encoding="utf-8", errors="ignore")
                else:
                    pages = []
                    with pdfplumber.open(pdf_path) as pdf:
                        for pg in pdf.pages:
                            pages.append(pg.extract_text() or "")
                    pdf_text = "\n---\n".join(pages)
                    txt_cache_path.write_text(pdf_text, encoding="utf-8")

                pages_list = pdf_text.split("\n---\n")

                # Find the scheme's page
                page_text = None
                for page in pages_list:
                    if _overlap(scheme_name, page[:1500]) >= 0.8:
                        page_text = page
                        break

                if not page_text:
                    continue

                # Extract facts using Qwen 3B
                got = qwen_extract(
                    page_text, 
                    FACTSHEET_SCHEMA,
                    f"From this mutual fund factsheet page for '{scheme_name}', extract the fields."
                )

                checks = {
                    "exit_load_pct": (0, 5), "exit_load_days": (0, 1500), "ytm_pct": (0, 20),
                    "modified_duration_yrs": (0, 40), "macaulay_duration_yrs": (0, 40),
                    "avg_maturity_yrs": (0, 50), "aum_cr": (0, 500000)
                }
                for k, (lo, hi) in checks.items():
                    if got.get(k) is not None and not (lo <= got[k] <= hi):
                        got[k] = None

                # Cache extracted facts to disk
                facts_cache_file = cache_path("factsheet_facts", f"{scheme_slug}_{m:%Y_%m}.json")
                facts_cache_file.write_text(
                    json.dumps({"data": got, "source_url": url}, indent=2), 
                    encoding="utf-8"
                )

                return fact(got, url, m)
            except Exception as e:
                log.warning("Factsheet failed for %s (%s): %s", scheme_name, url, e)

    return missing(f"https://{domain}", "factsheet page for this scheme not found")

In [ ]:
# ============================== SMART BENCHMARK RESOLVER ==============================
SEBI_CATEGORY_BENCHMARKS = {
    "small cap": "NIFTY SMALLCAP 250",
    "mid cap": "NIFTY MIDCAP 150",
    "large & mid cap": "NIFTY LARGEMIDCAP 250",
    "large cap": "NIFTY 100",
    "flexi cap": "NIFTY 500",
    "multi cap": "NIFTY 500",
    "focused": "NIFTY 500",
    "elss": "NIFTY 500",
    "value": "NIFTY 500",
    "contra": "NIFTY 500",
    "dividend yield": "NIFTY 500",
    "arbitrage": "NIFTY 50",       # or cash/arbitrage index
    "aggressive hybrid": "NIFTY 50",
    "balanced advantage": "NIFTY 50",
    "equity savings": "NIFTY 50",
}

def resolve_smart_benchmark(raw_bench: str | None, sub_category: str | None) -> tuple[str, bool]:
    """
    Infers the closest supported Nifty index from:
    1. Raw benchmark string (from AMFI or factsheet, including BSE conversions)
    2. SEBI sub-category mandate
    Returns: (canonical_index_name, is_fallback)
    """
    if raw_bench:
        b = str(raw_bench).upper()
        # Handle BSE / Nifty variations and map to canonical niftyindices names
        if "SMALL" in b:
            return "NIFTY SMALLCAP 250", False
        if "MID" in b and "LARGE" in b:
            return "NIFTY LARGEMIDCAP 250", False
        if "MID" in b:
            return "NIFTY MIDCAP 150", False
        if any(x in b for x in ("500", "200", "BSE 500", "S&P BSE 500")):
            return "NIFTY 500", False
        if any(x in b for x in ("100", "NEXT 50", "BSE 100")):
            return "NIFTY 100", False
        if any(x in b for x in ("SENSEX", "NIFTY 50", "BSE 30")):
            return "NIFTY 50", False
        if "BANK" in b:
            return "NIFTY BANK", False
        if "IT" in b or "TECH" in b:
            return "NIFTY IT", False

    # Fallback to SEBI category mandate
    if sub_category:
        sub = sub_category.lower()
        for cat_key, target_index in SEBI_CATEGORY_BENCHMARKS.items():
            if cat_key in sub:
                return target_index, False

    # Default fallback
    return DEFAULT_BENCHMARK, True

In [ ]:
# ---------- Agent 2 orchestrator: fetch everything o1 needs, once ----------
EQUITY_ORIENTED_HYBRIDS = ("aggressive hybrid", "balanced advantage", "dynamic asset allocation",
                           "arbitrage", "equity savings")
_perf_memo: dict = {}

def _find_code(meta: pd.DataFrame, isin: str, amfi_code: str | None) -> str | None:
    hit = meta.index[(meta["isin_growth"] == isin) | (meta["isin_reinv"] == isin)]
    if len(hit):
        return hit[0]
    return amfi_code if amfi_code in meta.index else None

def _peer_codes(meta: pd.DataFrame, category: str | None) -> list[str]:
    if not category:
        return []
    same = meta[meta["category"] == category]
    mask = same["name"].str.contains(r"direct", case=False, na=False) & ~same["name"].str.contains(
        r"idcw|dividend|bonus|payout", case=False, na=False)
    return list(same.index[mask])

def run_agent2(o1: list[dict]) -> dict:
    """Fetch everything the analysis needs, once, before Claude runs (linear flow, no loop)."""
    market = {"funds": {}, "tri": {}}
    meta = fetch_scheme_meta()
    codes = set()
    off = lambda: missing("", "look-through module switched off (ENABLE_AMC_DOCS = False)")
    for f in o1:
        info = market["funds"][f["isin"]] = {}
        info["code"] = _find_code(meta, f["isin"], f.get("amfi_code"))
        cat = meta.at[info["code"], "category"] if info["code"] else None
        info["asset_class"], info["sub_category"] = classify_category(cat)
        if info["asset_class"] == "hybrid" and any(w in info["sub_category"].lower() for w in EQUITY_ORIENTED_HYBRIDS):
            info["tax_class"] = "equity"
        else:
            info["tax_class"] = "equity" if info["asset_class"] == "equity" else "debt"

        # --- NEW SMART RESOLUTION ---
# 1. Manual override from BENCHMARK_MAP (if specified by you)
        if f["isin"] in BENCHMARK_MAP:
            info["benchmark"] = BENCHMARK_MAP[f["isin"]]
            info["benchmark_defaulted"] = False
        else:
            # 2. Extract benchmark reported by AMFI Performance data
            raw_bench = None
            perf_val = info.get("fund_performance", {}).get("value") or []
            if perf_val:
                matched_amfi = max(perf_val, key=lambda r: _overlap(f["scheme_name"], r.get("scheme", "") or ""), default=None)
                if matched_amfi and matched_amfi.get("benchmark"):
                    raw_bench = matched_amfi["benchmark"]
        
            # 3. Extract benchmark from Factsheet (if AMFI didn't report it)
            if not raw_bench and info.get("factsheet", {}).get("status") == "ok":
                raw_bench = info["factsheet"]["value"].get("benchmark")
        
            # 4. Resolve to canonical index
            resolved_bench, is_defaulted = resolve_smart_benchmark(raw_bench, info["sub_category"])
            info["benchmark"] = resolved_bench
            info["benchmark_defaulted"] = is_defaulted
    
        info["peers"] = [c for c in _peer_codes(meta, cat) if c != info["code"]][:MAX_PEERS]
        codes |= {c for c in [info["code"], *info["peers"]] if c}
        key = (info["asset_class"], info["sub_category"])
        if key not in _perf_memo:
            _perf_memo[key] = fetch_fund_performance(*key)
        info["fund_performance"] = _perf_memo[key]
        info["ter"] = fetch_ter(f["scheme_name"])
        info["holdings"] = fetch_holdings(f["scheme_name"], f["amc"]) if ENABLE_AMC_DOCS else off()
        info["factsheet"] = fetch_factsheet_facts(f["scheme_name"], f["amc"]) if ENABLE_AMC_DOCS else off()
    log.info("Agent 2: fetching NAV history for %d schemes (your funds + peers)", len(codes))
    market["panel"], market["meta"] = build_month_end_panel(sorted(codes)), meta
    for idx in {i["benchmark"] for i in market["funds"].values()}:
        market["tri"][idx] = fetch_tri(idx)
    market["rates"], market["macro"] = static_rates_and_macro()
    market["macro"].update({"pe_pb": fetch_pe_pb(MARKET_INDEX), "pe_pb_mid": fetch_pe_pb("NIFTY MIDCAP 150"),
                            "pe_pb_small": fetch_pe_pb("NIFTY SMALLCAP 100")})
    return market

def data_status(market: dict) -> list[dict]:
    """Everything that is missing, so the report can say so instead of guessing."""
    out = []
    for isin, info in market["funds"].items():
        for k in ("fund_performance", "ter", "holdings", "factsheet"):
            if info.get(k, {}).get("status") != "ok":
                out.append({"isin": isin, "item": k, "reason": info.get(k, {}).get("reason", "not fetched")})
        if info.get("benchmark_defaulted"):
            out.append({"isin": isin, "item": "benchmark", "reason": f"not in BENCHMARK_MAP, used {DEFAULT_BENCHMARK}"})
    for idx, f in market["tri"].items():
        if f["status"] != "ok":
            out.append({"isin": None, "item": f"tri:{idx}", "reason": f["reason"]})
    for k, f in {**market["rates"], **market["macro"]}.items():
        if f["status"] != "ok" or "STALE" in (f.get("reason") or ""):
            out.append({"isin": None, "item": k, "reason": f["reason"]})
    return out

## Metrics node (local code, real values)
Computes every number your framework asks for. Runs on real rupee values on your machine, so
tax thresholds (₹1.25L exemption, slab rates) are applied correctly before masking.
Naming rule used everywhere below: keys ending in `_inr` are rupees and keys ending in `_units`
are unit counts. The mask node divides exactly those by K.

In [ ]:
from scipy.optimize import brentq

PURCHASE_TYPES = {"PURCHASE", "PURCHASE_SIP", "SWITCH_IN", "SWITCH_IN_MERGER", "GIFT_IN"}
OUTFLOW_TYPES = {"REDEMPTION", "SWITCH_OUT", "SWITCH_OUT_MERGER", "GIFT_OUT"}

def xirr(flows: list[tuple[dt.date, float]]) -> float | None:
    """Money-weighted return. flows: (date, amount), investor outflows negative."""
    flows = sorted((pd.Timestamp(d).date(), a) for d, a in flows if a)
    if len(flows) < 2 or not (any(a < 0 for _, a in flows) and any(a > 0 for _, a in flows)):
        return None
    d0 = flows[0][0]
    f = lambda r: sum(a / (1 + r) ** ((d - d0).days / 365.0) for d, a in flows)
    try:
        return brentq(f, -0.99, 10)
    except ValueError:
        return None

def cash_flows(h: dict, start_nav: float | None) -> list[tuple]:
    flows = []
    if h.get("open_units", 0) > 0 and h["transactions"]:
        first = pd.Timestamp(h["transactions"][0]["date"]).date()
        nav0 = start_nav or h["transactions"][0]["nav"] or h["nav"]
        flows.append((first, -h["open_units"] * nav0))   # opening balance treated as invested at statement start
    for t in h["transactions"]:
        if t["amount"] is None:
            continue
        if t["type"] in PURCHASE_TYPES:
            flows.append((t["date"], -abs(t["amount"])))
        elif t["type"] in OUTFLOW_TYPES:
            flows.append((t["date"], abs(t["amount"])))
        elif t["type"] == "DIVIDEND_PAYOUT":
            flows.append((t["date"], abs(t["amount"])))
    return flows

def tri_on(tri: pd.Series, d) -> float:
    s = tri.loc[:pd.Timestamp(d)]
    return float(s.iloc[-1]) if len(s) else float(tri.iloc[0])

def ann_return(r: pd.Series) -> float:
    return float((1 + r).prod() ** (12 / len(r)) - 1) if len(r) else np.nan

def max_drawdown(r: pd.Series) -> float:
    path = (1 + r).cumprod()
    return float((path / path.cummax() - 1).min()) if len(r) else np.nan

def risk_metrics(fr: pd.Series, br: pd.Series, rf: float) -> dict:
    """Section 3 metrics on monthly returns (last 36-60 months available)."""
    df = pd.concat([fr, br], axis=1, keys=["f", "b"]).dropna().iloc[-60:]
    if len(df) < 24:
        return {"months_used": len(df), "status": "insufficient history (<24 months)"}
    f, b = df["f"], df["b"]
    af, ab = ann_return(f), ann_return(b)
    sd = f.std() * np.sqrt(12)
    dd = np.sqrt((np.minimum(f - rf / 12, 0) ** 2).mean()) * np.sqrt(12)
    beta = f.cov(b) / b.var()
    te = (f - b).std() * np.sqrt(12)
    up, dn = b > 0, b < 0
    mdd = max_drawdown(f)
    return {
        "months_used": len(df), "cagr_pct": af * 100, "bench_cagr_pct": ab * 100,
        "std_dev_pct": sd * 100, "downside_dev_pct": dd * 100, "beta": beta,
        "sharpe": (af - rf) / sd if sd else None, "sortino": (af - rf) / dd if dd else None,
        "treynor": (af - rf) / beta if beta else None,
        "jensen_alpha_pct": (af - (rf + beta * (ab - rf))) * 100,
        "tracking_error_pct": te * 100, "info_ratio": (af - ab) / te if te else None,
        "r_squared": f.corr(b) ** 2,
        "up_capture": f[up].mean() / b[up].mean() * 100 if up.any() else None,
        "down_capture": f[dn].mean() / b[dn].mean() * 100 if dn.any() else None,
        "max_drawdown_pct": mdd * 100, "calmar": af / abs(mdd) if mdd else None,
    }

def rolling_cagr(nav_m: pd.Series, months: int = 36) -> pd.Series:
    return (nav_m / nav_m.shift(months)) ** (12 / months) - 1

def rolling_stats(nav_m: pd.Series, tri_m: pd.Series) -> dict:
    rf_, rb_ = rolling_cagr(nav_m), rolling_cagr(tri_m)
    df = pd.concat([rf_, rb_], axis=1, keys=["f", "b"]).dropna()
    if df.empty:
        return {"status": "insufficient history for 3y rolling"}
    return {"rolling3y_mean_pct": df["f"].mean() * 100, "rolling3y_median_pct": df["f"].median() * 100,
            "rolling3y_min_pct": df["f"].min() * 100, "pct_windows_beating_bench": (df["f"] > df["b"]).mean() * 100,
            "windows": len(df)}

def quartile_history(panel: pd.DataFrame, code: str, peers: list[str]) -> dict:
    """Quartile of the fund's 3y rolling return among peers at each of the last 8 quarter ends."""
    cols = [c for c in set(peers) | {code} if c in panel.columns]
    roll = rolling_cagr(panel[cols])
    qs = [d for d in roll.index if d.month in (3, 6, 9, 12)][-8:]
    hist = []
    for d in qs:
        row = roll.loc[d].dropna()
        if code in row and len(row) >= 4:
            pct = (row < row[code]).mean()        # share of peers the fund beats
            hist.append({"quarter": str(d.date()), "quartile": 4 - min(3, int(pct * 4)), "pct_rank": pct * 100})
    streak = lambda cond: next((i for i, h in enumerate(reversed(hist)) if not cond(h)), len(hist))
    return {"history": hist,
            "bottom_quartile_streak": streak(lambda h: h["quartile"] == 4),
            "below_median_streak": streak(lambda h: h["pct_rank"] < 50)}

def is_debt_row(r: dict) -> bool:
    txt = f"{r.get('industry_or_rating', '')} {r.get('instrument', '')}"
    return bool(re.search(r"\b(AAA|AA[+-]?|A1\+|SOV|SOVEREIGN|CRISIL|ICRA|CARE|IND A|T-?BILL|TREASURY|G-?SEC|"
                          r"GOVERNMENT|TREPS|REVERSE REPO|CASH|NET (CURRENT )?ASSETS|DEBENTURE|NCD|CERTIFICATE OF DEPOSIT|COMMERCIAL PAPER)\b",
                          txt, re.I))

def fifo_lots(h: dict, tax_class: str, exit_rule: tuple[int, float], today: dt.date) -> dict:
    """Section 2D: unrealised gains by lot (FIFO), tax buckets and exit-load ageing."""
    lots = []
    if h.get("open_units", 0) > 0:
        first = h["transactions"][0] if h["transactions"] else None
        lots.append({"date": pd.Timestamp(first["date"]).date() if first else today, "units": h["open_units"],
                     "nav": (first or {}).get("nav") or h["nav"], "approx": True})
    fy_start = dt.date(today.year if today.month >= 4 else today.year - 1, 4, 1)
    realised_eq_ltcg_fy = 0.0
    for t in h["transactions"]:
        u = t["units"] or 0
        if t["type"] in PURCHASE_TYPES or t["type"] == "DIVIDEND_REINVEST":
            if u > 0:
                lots.append({"date": pd.Timestamp(t["date"]).date(), "units": u, "nav": t["nav"] or 0, "approx": False})
        elif u < 0:
            sell, d = -u, pd.Timestamp(t["date"]).date()
            while sell > 1e-9 and lots:
                lot = lots[0]
                take = min(sell, lot["units"])
                if tax_class == "equity" and (d - lot["date"]).days > FRAMEWORK["equity_ltcg_days"] and d >= fy_start:
                    realised_eq_ltcg_fy += take * ((t["nav"] or 0) - lot["nav"])
                lot["units"] -= take
                sell -= take
                if lot["units"] <= 1e-9:
                    lots.pop(0)
    nav = h["nav"]
    load_days, load_rate = exit_rule
    stcg = ltcg = slab_gain = load_cost = free_value = 0.0
    for lot in lots:
        days = (today - lot["date"]).days
        gain = lot["units"] * (nav - lot["nav"])
        val = lot["units"] * nav
        if tax_class == "equity":
            if days > FRAMEWORK["equity_ltcg_days"]:
                ltcg += gain
            else:
                stcg += gain
        else:
            if lot["date"] >= dt.date(2023, 4, 1) or days <= 730:
                slab_gain += gain
            else:
                ltcg += gain   # pre-Apr-2023 debt held > 24 months: 12.5% (verify current rules)
        in_load = days <= load_days
        load_cost += val * load_rate if in_load else 0.0
        if not in_load and (tax_class != "equity" or days > FRAMEWORK["equity_ltcg_days"]):
            free_value += val
    return {"unrealised_stcg_inr": stcg, "unrealised_ltcg_inr": ltcg, "unrealised_slab_gain_inr": slab_gain,
            "exit_load_if_sold_now_inr": load_cost, "value_free_of_load_and_stcg_inr": free_value,
            "realised_equity_ltcg_this_fy_inr": realised_eq_ltcg_fy,
            "lots_count": len(lots), "opening_balance_approximated": any(l["approx"] for l in lots)}

def capacity_score(p: dict) -> float:
    s = 5.0
    s += 1.5 if p["age"] < 35 else 0.5 if p["age"] < 50 else -1.5
    s += 1 if p["income_stability"] == "stable" else -1
    s += 1 if p["emergency_fund_months"] >= 6 else -1.5
    s -= 1 if p["debt_to_income"] > 0.4 else 0
    s -= 0.5 * max(0, p["dependents"] - 1)
    return float(np.clip(s, 1, 10))

def detected_monthly_sip(o2: list[dict], today: dt.date) -> float:
    """Sum of SIP instalments in the last 45 days (one instalment per running SIP)."""
    cutoff = today - dt.timedelta(days=45)
    return float(sum(t["amount"] or 0 for h in o2 for t in h["transactions"]
                     if t["type"] == "PURCHASE_SIP" and dt.date.fromisoformat(t["date"]) >= cutoff))

def projection(corpus: float, sip: float, years: int, alloc: dict, runs: int) -> dict:
    """Generic long-term goal: simulate where today's portfolio plus current SIPs could be in `years`."""
    rng = np.random.default_rng(42)
    mu = sum(w * FRAMEWORK["mc_assumptions"].get(a, (0.07, 0.05))[0] for a, w in alloc.items())
    var = sum((w * FRAMEWORK["mc_assumptions"].get(a, (0.07, 0.05))[1]) ** 2 for a, w in alloc.items())
    m = years * 12
    rets = rng.normal(mu / 12, np.sqrt(var / 12), size=(runs, m))   # assets treated as independent
    wealth = np.full(runs, corpus, dtype=float)
    for i in range(m):
        wealth = (wealth + sip) * (1 + rets[:, i])
    invested = corpus + sip * m
    p10, p50, p90 = np.percentile(wealth, [10, 50, 90])
    return {"years": years, "monthly_sip_inr": sip, "total_invested_by_then_inr": invested,
            "p10_value_inr": p10, "median_value_inr": p50, "p90_value_inr": p90,
            "prob_below_amount_invested_pct": float((wealth < invested).mean() * 100),
            "assumed_return_pct": mu * 100, "assumed_volatility_pct": float(np.sqrt(var)) * 100}

def percentile_rank(value, peer_values, higher_is_better=True) -> float | None:
    vals = [v for v in peer_values if v is not None and not pd.isna(v)]
    if value is None or pd.isna(value) or len(vals) < 3:
        return None
    pct = np.mean([v < value for v in vals]) * 100
    return pct if higher_is_better else 100 - pct

In [ ]:
def compute_metrics(o1: list[dict], o2: list[dict], market: dict) -> dict:
    today = dt.date.today()
    panel = market["panel"]
    rets = panel.pct_change(fill_method=None)
    rf_f = market["rates"]["tbill_91d"]
    rf = (rf_f["value"] / 100) if rf_f["status"] == "ok" else RISK_FREE_FALLBACK
    tri_m = {k: v["value"].resample("ME").last() for k, v in market["tri"].items() if v["status"] == "ok"}

    # ----- per holding (aggregate folios of the same ISIN) -----
    by_isin: dict[str, dict] = {}
    for h in o2:
        b = by_isin.setdefault(h["isin"], {"value": 0.0, "cost": 0.0, "units": 0.0, "parts": []})
        b["value"] += h["value"]; b["cost"] += h["cost"] or 0; b["units"] += h["units"]; b["parts"].append(h)
    total = sum(b["value"] for b in by_isin.values())
    o1_by = {f["isin"]: f for f in o1}

    funds, all_flows, bench_terminal, port_lot = [], [], 0.0, []
    for isin, b in by_isin.items():
        info, f1 = market["funds"][isin], o1_by[isin]
        code = info["code"]
        fs = info["factsheet"]["value"] if info["factsheet"]["status"] == "ok" else {}
        default_load = FRAMEWORK["default_exit_load"].get(info["asset_class"], (365, 0.01))
        exit_rule = ((fs.get("exit_load_days") or default_load[0]),
                     ((fs.get("exit_load_pct") or default_load[1] * 100) / 100))
        start_nav = None
        flows = []
        lots = {"unrealised_stcg_inr": 0, "unrealised_ltcg_inr": 0, "unrealised_slab_gain_inr": 0,
                "exit_load_if_sold_now_inr": 0, "value_free_of_load_and_stcg_inr": 0,
                "realised_equity_ltcg_this_fy_inr": 0, "lots_count": 0, "opening_balance_approximated": False}
        for h in b["parts"]:
            flows += cash_flows(h, start_nav)
            l = fifo_lots(h, info["tax_class"], exit_rule, today)
            for k, v in l.items():
                lots[k] = (lots[k] or v) if isinstance(v, bool) else lots[k] + v
        nav_date = pd.Timestamp(b["parts"][0]["nav_date"]).date()
        fund_x = xirr(flows + [(nav_date, b["value"])])
        all_flows += flows
        # Benchmark-equivalent: same cash flows on the same dates into the fund's benchmark TRI
        tri_d = market["tri"].get(info["benchmark"], {}).get("value") if market["tri"].get(info["benchmark"], {}).get("status") == "ok" else None
        if tri_d is not None:
            bu = 0.0
            for d, a in flows:
                bu += -a / tri_on(tri_d, d)
            bench_terminal += max(0.0, bu) * float(tri_d.iloc[-1])

        m = {"isin": isin, "scheme": f1["scheme_name"], "amc": f1["amc"], "plan": f1["plan"], "option": f1["option"],
             "asset_class": info["asset_class"], "sub_category": info["sub_category"], "tax_class": info["tax_class"],
             "benchmark": info["benchmark"], "benchmark_defaulted": info["benchmark_defaulted"],
             "value_inr": b["value"], "cost_inr": b["cost"], "held_units": b["units"], "weight_pct": b["value"] / total * 100,
             "xirr_pct": fund_x * 100 if fund_x is not None else None, "tax_and_load": lots,
             "exit_load_rule": {"days": exit_rule[0], "pct": exit_rule[1] * 100}}
        if code and code in panel.columns and info["benchmark"] in tri_m:
            fr = rets[code]
            br = tri_m[info["benchmark"]].pct_change().reindex(panel.index)
            m["risk"] = risk_metrics(fr, br, rf)
            m["rolling"] = rolling_stats(panel[code], tri_m[info["benchmark"]].reindex(panel.index))
            m["quartiles"] = quartile_history(panel, code, info["peers"])
            # Peer metrics for percentile ranks (same SEBI sub-category, direct growth plans)
            peer_rows = []
            for pc in info["peers"]:
                if pc in panel.columns and pc != code:
                    pr = risk_metrics(rets[pc], br, rf)
                    if "sortino" in pr:
                        pr["pct_beat"] = rolling_stats(panel[pc], tri_m[info["benchmark"]].reindex(panel.index)).get("pct_windows_beating_bench")
                        peer_rows.append(pr)
            m["peer_count"] = len(peer_rows)
            m["_peer_rows"] = peer_rows
        else:
            m["risk"] = {"status": "missing NAV or benchmark history"}
        perf = info["fund_performance"]
        if perf["status"] == "ok":
            mine = max(perf["value"], key=lambda r: _overlap(f1["scheme_name"], r["scheme"] or ""), default=None)
            m["amfi_trailing"] = mine
            m["amfi_peer_trailing"] = perf["value"]
        ter = info["ter"]["value"] if info["ter"]["status"] == "ok" else {}
        m["ter_pct"] = ter.get("direct_pct") if f1["plan"] == "direct" else ter.get("regular_pct")
        m["ter_direct_pct"], m["ter_regular_pct"] = ter.get("direct_pct"), ter.get("regular_pct")
        m["factsheet"] = fs
        hold = info["holdings"]["value"] if info["holdings"]["status"] == "ok" else None
        m["_holdings"] = hold
        if hold:
            debt_w = sum(r["weight"] for r in hold if is_debt_row(r))
            m["lookthrough_equity_pct"] = max(0.0, 100 - debt_w) if info["asset_class"] in ("equity", "hybrid") else 0.0
            m["top10_weight_pct"] = sum(sorted((r["weight"] for r in hold), reverse=True)[:10])
        funds.append(m)

    # ----- portfolio return measures (2A) -----
    port_x = xirr(all_flows + [(today, total)])
    bench_x = xirr(all_flows + [(today, bench_terminal)]) if bench_terminal else None
    codes = {f["isin"]: market["funds"][f["isin"]]["code"] for f in funds}
    w_now = {f["isin"]: f["value_inr"] / total for f in funds}
    usable = [i for i, c in codes.items() if c in rets.columns]
    R = rets[[codes[i] for i in usable]].dropna().iloc[-60:]
    w = np.array([w_now[i] for i in usable]); w = w / w.sum() if w.sum() else w
    port_r = R @ w if len(R) else pd.Series(dtype=float)
    twr = ann_return(port_r) if len(port_r) >= 12 else None
    cov = R.cov() * 12 if len(R) >= 24 else None
    port = {
        "total_value_inr": total, "total_cost_inr": sum(f["cost_inr"] for f in funds),
        "xirr_pct": port_x * 100 if port_x is not None else None,
        "benchmark_equivalent_xirr_pct": bench_x * 100 if bench_x is not None else None,
        "twr_current_weights_pct": twr * 100 if twr is not None else None,
        "behaviour_gap_pct": (twr - port_x) * 100 if twr is not None and port_x is not None else None,
        "absolute_return_pct": (total / max(1, sum(f["cost_inr"] for f in funds)) - 1) * 100,
        "volatility_pct": float(np.sqrt(w @ cov.values @ w)) * 100 if cov is not None else None,
        "beta": sum(w_now[f["isin"]] * f["risk"].get("beta", np.nan) for f in funds if "beta" in f["risk"]) or None,
        "max_drawdown_pct": max_drawdown(port_r) * 100 if len(port_r) else None,
        "var95_monthly_pct": float(np.percentile(port_r, 5)) * 100 if len(port_r) >= 24 else None,
        "cvar95_monthly_pct": float(port_r[port_r <= np.percentile(port_r, 5)].mean()) * 100 if len(port_r) >= 24 else None,
        "correlation": R.corr().round(2).rename(index=dict(zip([codes[i] for i in usable], usable)),
                                                columns=dict(zip([codes[i] for i in usable], usable))).to_dict() if len(R) >= 24 else None,
        "notes": ["TWR uses current weights over the last 60 months (approximation).",
                  "Monte Carlo treats asset classes as independent."],
    }

    # ----- allocation, overlap, concentration (2B) -----
    alloc = {}
    for f in funds:
        eq = f.get("lookthrough_equity_pct")
        if f["asset_class"] == "hybrid":
            eq = eq if eq is not None else 50.0
            alloc["equity"] = alloc.get("equity", 0) + f["weight_pct"] * eq / 100
            alloc["debt"] = alloc.get("debt", 0) + f["weight_pct"] * (100 - eq) / 100
        else:
            a = f["asset_class"] if f["asset_class"] in ("equity", "debt", "gold", "intl") else "equity"
            alloc[a] = alloc.get(a, 0) + f["weight_pct"]
    port["allocation_pct"] = alloc
    port["category_mix_pct"] = pd.Series({f["sub_category"]: f["weight_pct"] for f in funds}).groupby(level=0).sum().to_dict()
    hold_w = {f["isin"]: {r["isin"]: r["weight"] for r in f["_holdings"]} for f in funds if f.get("_holdings")}
    overlaps = []
    keys = list(hold_w)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b2 = hold_w[keys[i]], hold_w[keys[j]]
            ov = sum(min(a[s], b2[s]) for s in set(a) & set(b2)) / 100
            overlaps.append({"fund_a": keys[i], "fund_b": keys[j], "overlap": ov,
                             "redundant": ov > FRAMEWORK["overlap_redundant"]})
    port["overlaps"] = overlaps
    stock_exp, sector_exp = {}, {}
    for f in funds:
        for r in f.get("_holdings") or []:
            stock_exp[r["instrument"] or r["isin"]] = stock_exp.get(r["instrument"] or r["isin"], 0) + f["weight_pct"] / 100 * r["weight"]
            sec = str(r.get("industry_or_rating") or "Unclassified")
            sector_exp[sec] = sector_exp.get(sec, 0) + f["weight_pct"] / 100 * r["weight"]
    hhi = lambda ws: sum((x / 100) ** 2 for x in ws)
    amc_w = pd.Series({f["amc"]: f["weight_pct"] for f in funds}).groupby(level=0).sum()
    port["top_stock_exposure_pct"] = dict(sorted(stock_exp.items(), key=lambda x: -x[1])[:15])
    port["top_sector_exposure_pct"] = dict(sorted(sector_exp.items(), key=lambda x: -x[1])[:10])
    port["concentration"] = {
        "fund_hhi": hhi(f["weight_pct"] for f in funds), "amc_hhi": hhi(amc_w.values),
        "stock_hhi": hhi(stock_exp.values()) if stock_exp else None,
        "sector_hhi": hhi(sector_exp.values()) if sector_exp else None,
        "effective_funds": 1 / hhi(f["weight_pct"] for f in funds),
        "amc_weights_pct": amc_w.to_dict(),
        "amc_over_limit": [a for a, x in amc_w.items() if x / 100 > FRAMEWORK["amc_concentration_max"]],
        "fund_count": len(funds),
    }

    # ----- costs and tax (2D) -----
    port["weighted_ter_pct"] = sum(f["weight_pct"] / 100 * (f["ter_pct"] or 0) for f in funds)
    horizon = PROFILE["horizon_years"]
    drag = 0.0
    for f in funds:
        if f["plan"] == "regular" and f["ter_regular_pct"] and f["ter_direct_pct"]:
            d = (f["ter_regular_pct"] - f["ter_direct_pct"]) / 100
            drag += f["value_inr"] * ((1.10) ** horizon - (1.10 - d) ** horizon)
    port["regular_plan_drag_inr"] = drag
    eq_ltcg = sum(f["tax_and_load"]["unrealised_ltcg_inr"] for f in funds if f["tax_class"] == "equity")
    realised = sum(f["tax_and_load"]["realised_equity_ltcg_this_fy_inr"] for f in funds)
    port["tax"] = {
        "rules_as_of": FRAMEWORK["tax_rules_as_of"],
        "equity_ltcg_exemption_inr": FRAMEWORK["equity_ltcg_exemption"],
        "ltcg_harvest_room_this_fy_inr": max(0.0, FRAMEWORK["equity_ltcg_exemption"] - realised),
        "unrealised_equity_ltcg_inr": eq_ltcg,
        "tax_if_everything_sold_inr": (max(0, eq_ltcg - max(0, FRAMEWORK["equity_ltcg_exemption"] - realised)) * FRAMEWORK["equity_ltcg_rate"]
                                       + sum(max(0, f["tax_and_load"]["unrealised_stcg_inr"]) for f in funds if f["tax_class"] == "equity") * FRAMEWORK["equity_stcg_rate"]
                                       + sum(max(0, f["tax_and_load"]["unrealised_slab_gain_inr"]) for f in funds) * FRAMEWORK["debt_slab_rate"]
                                       + sum(max(0, f["tax_and_load"]["unrealised_ltcg_inr"]) for f in funds if f["tax_class"] == "debt") * FRAMEWORK["equity_ltcg_rate"]),
    }

    # ----- client profile, generic long-term goal, target allocation (1, 6) -----
    cap = capacity_score(PROFILE)
    tol = PROFILE["risk_tolerance_score"] - (2 if PROFILE["stopped_sip_in_2020_or_2022"] else 0)
    risk_score = min(cap, tol)
    exp_ret = sum(alloc.get(a, 0) / 100 * FRAMEWORK["mc_assumptions"][a][0] for a in FRAMEWORK["mc_assumptions"])
    horizon = PROFILE["horizon_years"]
    glide = next(e for yrs, e in FRAMEWORK["glide_path_equity"] if horizon <= yrs)
    target_eq = float(np.clip(glide * (0.7 + 0.06 * risk_score), 0.05, 0.90))
    target = {"equity": target_eq * 100, "debt": (1 - target_eq) * 100}
    gap = []
    for a in sorted(set(target) | set(alloc)):
        cur, tgt = alloc.get(a, 0.0), target.get(a, 0.0)
        gap.append({"asset_class": a, "current_pct": cur, "target_pct": tgt, "gap_pp": cur - tgt,
                    "outside_band": abs(cur - tgt) > FRAMEWORK["rebalance_band_abs"] * 100,
                    "gap_inr": (cur - tgt) / 100 * total})
    sip = PROFILE["monthly_sip"] if PROFILE["monthly_sip"] is not None else detected_monthly_sip(o2, today)
    alloc_w = {a: x / 100 for a, x in alloc.items()}
    target_w = {a: x / 100 for a, x in target.items()}
    client = {"goal": "best long-term returns for the risk profile (no specific goal amount)",
              "horizon_years": horizon, "capacity_score": cap, "tolerance_score": tol, "final_risk_score": risk_score,
              "glide_path_equity_pct": glide * 100, "expected_portfolio_return_pct": exp_ret * 100,
              "monthly_sip_inr": sip, "monthly_sip_detected_from_cas": PROFILE["monthly_sip"] is None,
              "projection_current_allocation": projection(total, sip, horizon, alloc_w, FRAMEWORK["mc_runs"]),
              "projection_target_allocation": projection(total, sip, horizon, target_w, FRAMEWORK["mc_runs"])}

    # ----- fund scorecard (quant 85%) and rule-based verdict suggestion (4) -----
    W = FRAMEWORK["scorecard_weights"]
    max_overlap = {}
    for o in overlaps:
        for k in ("fund_a", "fund_b"):
            max_overlap[o[k]] = max(max_overlap.get(o[k], 0), o["overlap"])
    for f in funds:
        r, peers = f["risk"], f.pop("_peer_rows", [])
        f.pop("_holdings", None)
        sc = {}
        if "sortino" in r and peers:
            sc["consistency"] = percentile_rank(f["rolling"].get("pct_windows_beating_bench"), [p.get("pct_beat") for p in peers])
            ra = [percentile_rank(r["sortino"], [p["sortino"] for p in peers]),
                  percentile_rank(r["info_ratio"], [p["info_ratio"] for p in peers]),
                  percentile_rank((r["up_capture"] or 0) - (r["down_capture"] or 0),
                                  [(p["up_capture"] or 0) - (p["down_capture"] or 0) for p in peers])]
            sc["risk_adjusted"] = float(np.nanmean([x for x in ra if x is not None])) if any(x is not None for x in ra) else None
            dn = [percentile_rank(r["max_drawdown_pct"], [p["max_drawdown_pct"] for p in peers]),
                  percentile_rank(r["down_capture"], [p["down_capture"] for p in peers], higher_is_better=False)]
            sc["downside"] = float(np.nanmean([x for x in dn if x is not None])) if any(x is not None for x in dn) else None
        sc["cost"] = None if f["ter_pct"] is None else float(np.clip(100 - f["ter_pct"] * 40, 0, 100))
        sc["fit"] = 100 - max_overlap.get(f["isin"], 0) * 100 if hold_w else None
        avail = {k: v for k, v in sc.items() if v is not None}
        wsum = sum(W[k] for k in avail)
        f["scorecard"] = {"components_0_100": sc,
                          "quant_score_0_100": sum(W[k] * v for k, v in avail.items()) / wsum if wsum else None,
                          "manager_weight": W["manager"], "note": "Manager factor (15%) is scored by Agent 3."}
        q = f.get("quartiles", {})
        reasons = []
        if q.get("bottom_quartile_streak", 0) >= FRAMEWORK["exit_bottom_quartile_quarters"]:
            reasons.append(f"bottom quartile on 3y rolling for {q['bottom_quartile_streak']} quarters")
        if max_overlap.get(f["isin"], 0) > FRAMEWORK["overlap_redundant"]:
            reasons.append("over 50% overlap with another held fund")
        if r.get("r_squared", 0) > FRAMEWORK["r2_closet_indexer"] and (r.get("info_ratio") or 0) < 0:
            reasons.append("R² above 0.9 with negative information ratio (possible closet indexer)")
        f["rule_verdict"] = ("Exit" if reasons else
                             "Watch" if q.get("below_median_streak", 0) >= FRAMEWORK["watch_below_median_quarters"] else "Hold")
        f["rule_verdict_reasons"] = reasons

    # ----- macro overlay inputs (5) -----
    macro = {}
    pe = market["macro"]["pe_pb"]
    if pe["status"] == "ok" and "pe" in pe["value"]:
        s = pe["value"]["pe"].dropna()
        macro["nifty_pe"], macro["nifty_pe_10y_avg"] = float(s.iloc[-1]), float(s.mean())
        if "pb" in pe["value"]:
            pb = pe["value"]["pb"].dropna()
            macro["nifty_pb"], macro["nifty_pb_10y_avg"] = float(pb.iloc[-1]), float(pb.mean())
        g = market["rates"]["gsec_10y"]
        if g["status"] == "ok":
            macro["earnings_yield_minus_gsec_pp"] = 100 / macro["nifty_pe"] - g["value"]
    for k, lab in (("pe_pb_mid", "midcap"), ("pe_pb_small", "smallcap")):
        f_ = market["macro"][k]
        if f_["status"] == "ok" and "pe" in f_["value"] and "nifty_pe" in macro:
            macro[f"{lab}_pe_premium_pct"] = (float(f_["value"]["pe"].dropna().iloc[-1]) / macro["nifty_pe"] - 1) * 100
    for k in ("repo", "tbill_91d", "gsec_10y"):
        macro[k + "_pct"] = market["rates"][k]["value"]
    macro["cpi_yoy_pct"] = market["macro"]["cpi"]["value"]
    macro["fii_dii_latest"] = market["macro"]["fii_dii"]["value"]
    macro["risk_free_used_pct"] = rf * 100
    macro["risk_free_is_fallback"] = rf_f["status"] != "ok"

    return {"as_of": str(today), "client": client, "portfolio": port, "funds": funds,
            "target_allocation_pct": target, "gap_table": gap, "macro": macro,
            "missing_data": data_status(market), "framework": {k: v for k, v in FRAMEWORK.items() if k != "mc_assumptions"}}

## Mask / unmask (privacy rule)
Rupees (`*_inr`) and unit counts (`*_units`) are divided by K before Claude sees anything, and
multiplied back by K at the end. Percentages, ratios, weights and returns are unchanged by K,
so the analysis loses nothing. Claude writes any amount in its text as `[[INR:<number>]]`, so
unmasking multiplies only amounts and never a percentage.

In [ ]:
def _is_scaled_key(k: str) -> bool:
    return k.endswith("_inr") or k.endswith("_units")

def scale(obj, factor: float):
    """Return a copy with every *_inr / *_units number multiplied by factor."""
    if isinstance(obj, dict):
        return {k: (v * factor if _is_scaled_key(k) and isinstance(v, (int, float)) and v is not None
                    else scale(v, factor)) for k, v in obj.items()}
    if isinstance(obj, list):
        return [scale(v, factor) for v in obj]
    return obj

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, (np.floating, float)):
        return None if (obj is None or np.isnan(obj) or np.isinf(obj)) else round(float(obj), 4)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (pd.Timestamp, dt.date)):
        return str(obj)
    return obj

def mask_for_claude(metrics: dict) -> dict:
    m = to_jsonable(metrics)
    for f in m["funds"]:
        f.pop("amfi_peer_trailing", None)   # long peer table: keep it local, Claude gets ranks instead
    # A publicly known rupee amount (the Rs 1.25 lakh LTCG exemption) divided by K would reveal K.
    # So Claude gets the harvest room as a percentage, and the exemption-based figures stay local.
    t = m["portfolio"]["tax"]
    room, ltcg = t.pop("ltcg_harvest_room_this_fy_inr"), t["unrealised_equity_ltcg_inr"]
    t["harvestable_share_of_unrealised_equity_ltcg_pct"] = (min(100.0, room / ltcg * 100) if ltcg and ltcg > 0 else None)
    for k in ("equity_ltcg_exemption_inr", "tax_if_everything_sold_inr"):
        t.pop(k, None)
    m["framework"].pop("equity_ltcg_exemption", None)
    return scale(m, 1 / K)

INR_TAG = re.compile(r"\[\[INR:\s*(-?[\d,]*\.?\d+)\s*\]\]")

def unmask_text(text: str) -> str:
    def rep(m):
        v = float(m.group(1).replace(",", "")) * K
        return f"₹{v:,.0f}"
    return INR_TAG.sub(rep, text or "")

def unmask_draft(draft: dict) -> dict:
    d = scale(copy.deepcopy(draft), K)
    def walk(o):
        if isinstance(o, dict):
            return {k: walk(v) for k, v in o.items()}
        if isinstance(o, list):
            return [walk(v) for v in o]
        return unmask_text(o) if isinstance(o, str) else o
    return walk(d)

## Agent 3: analyst (Claude via your Pro sign-in)
Claude has **no tools**: it cannot browse, read files or fetch data. Agent 2 has already fetched
everything up front (linear flow), so Claude works only from the masked data it is given and
discloses anything missing. It runs in an empty folder with no settings loaded, and returns JSON
that must match `ANALYST_SCHEMA`.

In [ ]:
import concurrent.futures

def _sync_call_claude(system: str, prompt: str, schema: dict) -> dict:
    """Runs Claude Agent SDK inside an isolated thread with a Windows Proactor loop."""
    async def _inner():
        workdir = DATA_DIR / "claude_sandbox"
        workdir.mkdir(exist_ok=True)
        opts = ClaudeAgentOptions(
            tools=[], system_prompt=system, model=CLAUDE_MODEL, cwd=str(workdir), setting_sources=[],
            max_turns=4, output_format={"type": "json_schema", "schema": schema},
            env={"CLAUDE_CODE_OAUTH_TOKEN": os.environ.get("CLAUDE_CODE_OAUTH_TOKEN", "")},
        )
        result = None
        async for msg in query(prompt=prompt, options=opts):
            if isinstance(msg, ResultMessage):
                result = msg
        if result is None or result.is_error:
            raise RuntimeError(f"Claude call failed: {getattr(result, 'errors', None) or getattr(result, 'result', None)}")
        if result.structured_output is not None:
            return result.structured_output
        text = result.result or ""
        return json.loads(text[text.find("{"): text.rfind("}") + 1])

    # Windows requires ProactorEventLoop to spawn subprocesses via IOCP
    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
        
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(_inner())
    finally:
        loop.close()

async def call_claude(system: str, prompt: str, schema: dict) -> dict:
    # Run synchronously in a thread so Jupyter's Selector loop doesn't block it
    return await asyncio.to_thread(_sync_call_claude, system, prompt, schema)

In [ ]:
# ============================== AGENT 3: ANALYST ==============================
import os
import json
import asyncio
import sys
from claude_agent_sdk import query, ClaudeAgentOptions, ResultMessage

ACTION_TYPES = ["redirect_sip", "switch", "harvest_ltcg", "book_loss", "sell", "consolidate",
                "regular_to_direct", "stp", "no_action"]

ANALYST_SCHEMA = {
    "type": "object",
    "properties": {
        "portfolio_health": {"type": "object", "properties": {
            "summary": {"type": "string"}, "health_score_0_10": {"type": "number"},
            "key_issues": {"type": "array", "items": {"type": "string"}}},
            "required": ["summary", "health_score_0_10", "key_issues"]},
        "fund_reviews": {"type": "array", "items": {"type": "object", "properties": {
            "isin": {"type": "string"}, "verdict": {"type": "string", "enum": ["Hold", "Watch", "Exit"]},
            "manager_score_0_100": {"type": "number"},
            "reasons": {"type": "array", "items": {"type": "string"}},
            "framework_rules": {"type": "array", "items": {"type": "string"}}},
            "required": ["isin", "verdict", "manager_score_0_100", "reasons", "framework_rules"]}},
        "macro_tilt": {"type": "object", "properties": {
            "equity_tilt_pp": {"type": "number"}, "rationale": {"type": "string"}},
            "required": ["equity_tilt_pp", "rationale"]},
        "actions": {"type": "array", "items": {"type": "object", "properties": {
            "order": {"type": "integer"}, "type": {"type": "string", "enum": ACTION_TYPES},
            "from_isin": {"type": ["string", "null"]}, "to": {"type": ["string", "null"]},
            "amount_inr": {"type": "number"},
            "expected_alpha_pct_pa": {"type": ["number", "null"]}, "years": {"type": ["number", "null"]},
            "rationale": {"type": "string"}, "framework_rule": {"type": "string"}},
            "required": ["order", "type", "from_isin", "to", "amount_inr", "expected_alpha_pct_pa",
                         "years", "rationale", "framework_rule"]}},
        "missing_data_disclosures": {"type": "array", "items": {"type": "string"}},
        "report_markdown": {"type": "string"},
    },
    "required": ["portfolio_health", "fund_reviews", "macro_tilt", "actions",
                 "missing_data_disclosures", "report_markdown"],
}

ANALYST_SYSTEM = """You are a mutual fund portfolio analyst for an Indian investor. Follow the user's framework exactly:
order of work: client profile -> target allocation -> diagnose -> gap analysis -> fund quality -> macro tilt -> action plan -> monitoring.
The investor has ONE generic goal: the best long-term returns for their risk profile (horizon and projections are in
client). There are no goal amounts, so do not invent any. Rules: the goal comes before the fund; asset allocation drives most outcomes; every recommendation must clear tax, exit load and
transaction friction (break-even: expected alpha gain x years > tax + exit load + time out of market); prefer 5-8 funds with low overlap;
macro views may shift allocation by at most +/-10 percentage points and never override the risk-based target allocation; least-cost order of actions:
redirect new SIPs first, then switch units past exit load and in long-term gain, then harvest LTCG within the exemption and book losses,
and only then sell units that trigger STCG. Fund verdicts: Hold = top half of category and fits its role; Watch = 2-4 quarters below
median or manager change; Exit = bottom quartile on 3y rolling returns for 4+ quarters, style drift, governance concern, or redundancy.

DATA RULES
- Use only numbers present in the input JSON. Never invent or estimate a number that is not there. If something you need is
  missing (null, or listed in missing_data), say so in missing_data_disclosures and lower your confidence accordingly.
- All rupee values (keys ending _inr) and unit counts (keys ending _units) are scaled by an undisclosed constant. Ratios,
  percentages and returns are exact. Treat scaled amounts as if they were real and keep them in the same scale.
- In any text field, write every rupee amount as [[INR:<number in the same scale>]], for example [[INR:1520.5]]. Never write ₹, Rs or
  INR followed by a number. Percentages are written normally.
- amount_inr in actions is in the same scale. Use 0 for actions without an amount.
- LTCG harvesting room is given as harvestable_share_of_unrealised_equity_ltcg_pct (the share of unrealised equity LTCG
  that can be booked tax-free this financial year). Size harvest_ltcg actions with it; never quote the exemption in rupees.
- Score each fund's manager, process and AMC quality (0-100) from the factsheet facts given; if they are missing, give 50 and say so.
- report_markdown: a clear report for the investor with sections: Portfolio health, Allocation and gaps, Fund-by-fund review,
  Macro view, Action plan (in least-cost order), Missing data, Monitoring plan. End with: "This is a methodology-based analysis,
  not personal advice. Check it with a SEBI-registered investment adviser before acting."
- If a previous critique is given, fix every issue it lists."""

def _run_claude_proactor(system: str, prompt: str, schema: dict) -> dict:
    """Runs Claude CLI inside a dedicated thread using Windows ProactorEventLoop to support subprocesses."""
    async def _runner():
        workdir = DATA_DIR / "claude_sandbox"
        workdir.mkdir(exist_ok=True)
        opts = ClaudeAgentOptions(
            tools=[], system_prompt=system, model=CLAUDE_MODEL, cwd=str(workdir), setting_sources=[],
            max_turns=4, output_format={"type": "json_schema", "schema": schema},
            env={"CLAUDE_CODE_OAUTH_TOKEN": os.environ.get("CLAUDE_CODE_OAUTH_TOKEN", "")},
        )
        result = None
        async for msg in query(prompt=prompt, options=opts):
            if isinstance(msg, ResultMessage):
                result = msg
        if result is None or result.is_error:
            raise RuntimeError(f"Claude call failed: {getattr(result, 'errors', None) or getattr(result, 'result', None)}")
        if result.structured_output is not None:
            return result.structured_output
        text = result.result or ""
        return json.loads(text[text.find("{"): text.rfind("}") + 1])

    # Windows requires ProactorEventLoop for subprocess pipes
    if sys.platform == "win32":
        loop = asyncio.ProactorEventLoop()
    else:
        loop = asyncio.new_event_loop()
        
    asyncio.set_event_loop(loop)
    try:
        return loop.run_until_complete(_runner())
    finally:
        loop.close()

async def call_claude(system: str, prompt: str, schema: dict) -> dict:
    # Run in a separate thread so it doesn't collide with Jupyter's Selector loop
    return await asyncio.to_thread(_run_claude_proactor, system, prompt, schema)

async def run_analyst(masked: dict, previous: dict | None, critique: dict | None) -> dict:
    parts = [f"INPUT (masked):\n{json.dumps(masked, ensure_ascii=False)}"]
    if previous:
        parts.append(f"PREVIOUS DRAFT:\n{json.dumps(previous, ensure_ascii=False)}")
    if critique:
        parts.append(f"CRITIQUE TO FIX:\n{json.dumps(critique, ensure_ascii=False)}")
    return await call_claude(ANALYST_SYSTEM, "\n\n".join(parts), ANALYST_SCHEMA)

## Critic (reflection): rule checks in code + local Qwen review
Code checks your framework's hard rules on the **unmasked** draft (real rupees, locally).
Qwen then reviews reasoning and framework fit. The draft goes back to Agent 3 with the issues
listed, up to `MAX_REVISIONS` times.

In [ ]:
SWITCH_FRICTION = 0.0025   # time out of market + transaction friction, as a share of the amount moved

def rule_checks(draft_real: dict, metrics: dict) -> list[str]:
    issues = []
    funds = {f["isin"]: f for f in metrics["funds"]}
    acts = sorted(draft_real.get("actions", []), key=lambda a: a["order"])
    for a in acts:
        if not a.get("framework_rule"):
            issues.append(f"Action {a['order']} ({a['type']}) does not cite a framework rule.")
        if a["type"] in ("switch", "sell", "consolidate", "regular_to_direct") and a.get("from_isin") in funds:
            f = funds[a["from_isin"]]
            frac = min(1.0, a["amount_inr"] / f["value_inr"]) if f["value_inr"] else 0
            tl = f["tax_and_load"]
            tax = frac * (max(0, tl["unrealised_stcg_inr"]) * FRAMEWORK["equity_stcg_rate"]
                          + max(0, tl["unrealised_ltcg_inr"]) * FRAMEWORK["equity_ltcg_rate"]
                          + max(0, tl["unrealised_slab_gain_inr"]) * FRAMEWORK["debt_slab_rate"])
            cost = tax + frac * tl["exit_load_if_sold_now_inr"] + SWITCH_FRICTION * a["amount_inr"]
            if a.get("expected_alpha_pct_pa") is None or a.get("years") is None:
                issues.append(f"Action {a['order']}: break-even test impossible, expected alpha or years missing.")
            else:
                gain = a["expected_alpha_pct_pa"] / 100 * a["years"] * a["amount_inr"]
                if gain <= cost:
                    issues.append(f"Action {a['order']} fails break-even: expected gain ₹{gain:,.0f} vs cost ₹{cost:,.0f} "
                                  f"(tax + exit load + friction).")
    redirect = [a["order"] for a in acts if a["type"] == "redirect_sip"]
    sells = [a["order"] for a in acts if a["type"] == "sell"]
    if redirect and sells and min(sells) < max(redirect):
        issues.append("Least-cost order broken: a sell comes before redirecting SIPs.")
    if abs(draft_real.get("macro_tilt", {}).get("equity_tilt_pp", 0)) > FRAMEWORK["macro_tilt_max"] * 100:
        issues.append("Macro tilt exceeds +/-10 percentage points.")
    for r in draft_real.get("fund_reviews", []):
        f = funds.get(r["isin"])
        if r["verdict"] == "Exit" and f and f["rule_verdict"] != "Exit":
            text = " ".join(r["reasons"]).lower()
            if not re.search(r"governance|drift|manager|merger|category|redundan|overlap", text):
                issues.append(f"Exit verdict for {f['scheme']} has no framework trigger in the data.")
    exits = sum(1 for r in draft_real.get("fund_reviews", []) if r["verdict"] == "Exit")
    remaining = metrics["portfolio"]["concentration"]["fund_count"] - exits
    lo, hi = FRAMEWORK["ideal_fund_count"]
    if not lo <= remaining <= hi and "fund count" not in draft_real.get("report_markdown", "").lower():
        issues.append(f"Portfolio would hold {remaining} funds (target {lo}-{hi}) and the report does not explain it.")
    if metrics["missing_data"] and not draft_real.get("missing_data_disclosures"):
        issues.append("Some data is missing but the draft does not disclose it.")
    if "sebi-registered" not in draft_real.get("report_markdown", "").lower():
        issues.append("Report must end with the SEBI-registered adviser note.")
    return issues

def untagged_amounts(draft_masked: dict) -> list[str]:
    """Run on the MASKED draft: any ₹/Rs/INR number not inside [[INR:...]] would be unmasked wrongly."""
    bad = []
    for txt in [draft_masked.get("report_markdown", "")] + [a.get("rationale", "") for a in draft_masked.get("actions", [])]:
        stripped = INR_TAG.sub("", txt)
        if re.search(r"(₹|\bRs\.?|\bINR)\s?\d", stripped):
            bad.append("Text contains a rupee amount not written as [[INR:...]].")
            break
    return bad

CRITIC_SCHEMA = {"type": "object", "properties": {"passed": {"type": "boolean"},
                 "issues": {"type": "array", "items": {"type": "string"}}}, "required": ["passed", "issues"]}

def qwen_review(draft_masked: dict, metrics_masked: dict) -> dict:
    import ollama
    brief = {
        "portfolio": {k: metrics_masked["portfolio"].get(k) for k in
                      ("xirr_pct", "benchmark_equivalent_xirr_pct", "allocation_pct", "overlaps", "weighted_ter_pct")},
        "gap_table": metrics_masked["gap_table"],
        "funds": [{k: f.get(k) for k in ("isin", "scheme", "rule_verdict", "rule_verdict_reasons", "scorecard")}
                  for f in metrics_masked["funds"]]
    }
    prompt = (
        "You are a strict reviewer. Check the DRAFT against the DATA and these rules: the goal comes before the fund; allocation "
        "first; every action must beat tax, exit load and friction; 5-8 low-overlap funds; macro tilt within +/-10pp; least-cost "
        "order (redirect SIPs, then switch long-term units past exit load, then harvest LTCG, sell STCG units last); Exit needs a "
        "framework trigger. List concrete issues: numbers in the draft that are not in the data, verdicts that contradict the "
        "data, recommendations without a rule, missing sections. passed=true only if there are no material issues.\n\n"
        f"DATA:\n{json.dumps(brief, ensure_ascii=False)[:6000]}\n\n"
        f"DRAFT:\n{json.dumps({k: draft_masked[k] for k in ('portfolio_health', 'fund_reviews', 'macro_tilt', 'actions')}, ensure_ascii=False)[:7000]}"
    )
    resp = ollama.chat(
        model=globals().get("OLLAMA_CRITIC_MODEL", "qwen2.5:7b"),
        messages=[{"role": "user", "content": prompt}],
        format=CRITIC_SCHEMA,
        options={"temperature": 0, "num_ctx": 12288}
    )
    return json.loads(resp.message.content)

## LangGraph: state, nodes, edges

In [ ]:
from langgraph.graph import StateGraph, START, END

class PortfolioState(TypedDict, total=False):
    o1: list           # Agent 1 -> Agent 2: fund identity only
    o2: list           # Agent 1 -> metrics: real holdings (local only)
    cas_meta: dict
    market: dict       # Agent 2 output (local)
    metrics: dict      # real values (local)
    masked: dict       # what Claude sees (÷K)
    draft: dict        # Agent 3 output (masked scale)
    critique: dict
    revisions: int
    report: str
    
def node_parse_cas(state: PortfolioState) -> dict:
    if not CAS_PASSWORD:
        raise ValueError("CAS_PASSWORD is missing or empty in your .env file.")
        
    o1, o2, meta = parse_cas(CAS_PDF_PATH, password=CAS_PASSWORD)
    log.info("Agent 1: %d funds, %d holdings parsed", len(o1), len(o2))
    return {"o1": o1, "o2": o2, "cas_meta": meta, "revisions": 0}

def node_fetch(state: PortfolioState) -> dict:
    return {"market": run_agent2(state["o1"])}

def node_metrics(state: PortfolioState) -> dict:
    return {"metrics": compute_metrics(state["o1"], state["o2"], state["market"])}

def node_mask(state: PortfolioState) -> dict:
    return {"masked": mask_for_claude(state["metrics"])}

async def node_analyst(state: PortfolioState) -> dict:
    return {"draft": await run_analyst(state["masked"], state.get("draft"), state.get("critique"))}

def node_critic(state: PortfolioState) -> dict:
    draft_real = unmask_draft(state["draft"])
    issues = untagged_amounts(state["draft"]) + rule_checks(draft_real, state["metrics"])
    try:
        llm = qwen_review(state["draft"], state["masked"])
    except Exception as e:
        llm = {"passed": True, "issues": [], "note": f"Qwen review skipped: {e}"}
    issues += llm.get("issues", []) if not llm.get("passed", True) else []
    return {"critique": {"passed": not issues, "issues": issues},
            "revisions": state.get("revisions", 0) + (1 if issues else 0)}

def route_after_critic(state: PortfolioState) -> str:
    if state["critique"]["passed"] or state.get("revisions", 0) > MAX_REVISIONS:
        return "finalize"
    return "analyst"

def node_finalize(state: PortfolioState) -> dict:
    d = unmask_draft(state["draft"])
    m = state["metrics"]
    W = FRAMEWORK["scorecard_weights"]
    mgr = {r["isin"]: r["manager_score_0_100"] for r in d.get("fund_reviews", [])}
    lines = [d["report_markdown"], "\n---\n## Appendix (computed locally from your data)\n",
             f"As of {m['as_of']}. Tax rules as of {FRAMEWORK['tax_rules_as_of']}: verify current rules before acting.\n",
             "| Fund | Plan | Value | Weight | XIRR | Quant score | Manager | Final score | Rule verdict |",
             "| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | --- |"]
    for f in m["funds"]:
        q = f["scorecard"]["quant_score_0_100"]
        final = (q * (1 - W["manager"]) + mgr.get(f["isin"], 50) * W["manager"]) if q is not None else None
        xirr_s = "" if f["xirr_pct"] is None else f"{f['xirr_pct']:.1f}%"
        q_s = "" if q is None else f"{q:.0f}"
        final_s = "" if final is None else f"{final:.0f}"
        lines.append(f"| {f['scheme']} | {f['plan']} | ₹{f['value_inr']:,.0f} | {f['weight_pct']:.1f}% | "
                     f"{xirr_s} | {q_s} | {mgr.get(f['isin'], '')} | {final_s} | {f['rule_verdict']} |")
    lines += ["", "| Asset class | Current | Target | Gap | Outside band |", "| --- | ---: | ---: | ---: | --- |"]
    for g in m["gap_table"]:
        lines.append(f"| {g['asset_class']} | {g['current_pct']:.1f}% | {g['target_pct']:.1f}% | {g['gap_pp']:+.1f} pp | {'yes' if g['outside_band'] else 'no'} |")
    t = m["portfolio"]["tax"]
    lines += ["", f"Unrealised equity LTCG: ₹{t['unrealised_equity_ltcg_inr']:,.0f}. "
                  f"LTCG harvest room this FY: ₹{t['ltcg_harvest_room_this_fy_inr']:,.0f}. "
                  f"Estimated tax if everything were sold: ₹{t['tax_if_everything_sold_inr']:,.0f}."]
    c = m["client"]
    for label, key in (("current allocation", "projection_current_allocation"), ("target allocation", "projection_target_allocation")):
        pj = c[key]
        lines += ["", f"Projection in {pj['years']} years at {label} (with ₹{pj['monthly_sip_inr']:,.0f}/month SIP, "
                      f"assumed {pj['assumed_return_pct']:.1f}% a year): median ₹{pj['median_value_inr']:,.0f}, "
                      f"bad case (10th pct) ₹{pj['p10_value_inr']:,.0f}, good case (90th pct) ₹{pj['p90_value_inr']:,.0f}."]
    if m["missing_data"]:
        lines += ["", "**Missing data:**"] + [f"- {x['item']} {x['isin'] or ''}: {x['reason']}" for x in m["missing_data"]]
    if state.get("critique") and not state["critique"]["passed"]:
        lines += ["", "**Unresolved reviewer notes:**"] + [f"- {i}" for i in state["critique"]["issues"]]
    report = "\n".join(lines)
    REPORT_PATH.write_text(report, encoding="utf-8")
    return {"report": report}

g = StateGraph(PortfolioState)
g.add_node("parse_cas", node_parse_cas)      # Agent 1
g.add_node("fetch", node_fetch)              # Agent 2
g.add_node("metrics", node_metrics)
g.add_node("mask", node_mask)
g.add_node("analyst", node_analyst)          # Agent 3 (Claude)
g.add_node("critic", node_critic)            # reflection (code + Qwen)
g.add_node("finalize", node_finalize)        # unmask x K + report
g.add_edge(START, "parse_cas")
g.add_edge("parse_cas", "fetch")
g.add_edge("fetch", "metrics")
g.add_edge("metrics", "mask")
g.add_edge("mask", "analyst")
g.add_edge("analyst", "critic")
g.add_conditional_edges("critic", route_after_critic, {"analyst": "analyst", "finalize": "finalize"})
g.add_edge("finalize", END)
app = g.compile()

## Graph picture
Drawn from the compiled graph itself. `draw_mermaid_png` sends only the node names to the
mermaid.ink web service to render the image; if that is unavailable, an ASCII drawing is shown.

In [ ]:
from IPython.display import Image, Markdown, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(PNG rendering unavailable: {e})\n")
    try:
        print(app.get_graph().draw_ascii())
    except Exception:
        print(app.get_graph().draw_mermaid())

## Run

In [ ]:
final_state = await app.ainvoke({}, config={"recursion_limit": 40})
display(Markdown(final_state["report"]))
print(f"Saved to {REPORT_PATH.resolve()}")

## Optional: monthly re-run without opening Jupyter
Install `papermill` (`python -m pip install papermill`) and schedule this command with Windows
Task Scheduler or cron: `papermill mf_portfolio_analyser.ipynb mf_run_output.ipynb`.
Cached NAV files make later runs much faster.

Very high latency since each factsheet pdf is being downloaded from AMFI server on a weekend (amfi response is already very slow, weekend makes the script look some days backward to fetch the most recent data)
Also, pdf parsing is somewhat relatively more difficult for the LLM to process with more accuracy issues as well


Hence, using a pdf to markdown converter + qwen 3b instead of 7b - since both work same for this particular task

In [ ]:
import sys
!{sys.executable} -m pip install pymupdf4llm